In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.nn import functional as F

import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from matplotlib import pyplot as plt

In [ ]:
# 设置随机数seed
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [ ]:
# 设置机器，cuda或者cpu
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using {device}")

# 0. 大语言模型简介

大语言模型，一言以蔽之，是在做这样一件事情，基于给定的一串“文字”猜测下一个“文字”是什么，然后循环这个过程，直至遇到结束条件（如模型生成出约定的结束符 EOS，又或已经生成到了最长可容许长度）。而称之为“大”，则说其模型尺寸“大”、训练数据“大”。

对于语言模型而言，给定的“文字”被唤作 token，可以是我们平日里用的单词或者字符，亦可以是其他的构成语言要素的东西。即是，给定模型一串 tokens，模型基于给定信息预测下一个 token（Next Token Prediction）。写成公式如下：

$$
p(t_{k} | (t_{1}, t_{2}, ..., t_{k-1}), \theta)
$$

其中，条件侧 $t_{1}, t_{2}, ..., t_{k-1}$ 代表前 $k-1$ 个 tokens，$\theta$ 代表模型，而另一边 $t_{k}$ 代表预测的第 $k$ 个 token；至于 $p(\cdot)$，则是模型为这第 $k$ 个 token 开出的概率——故所谓“猜测”，实为“按概率猜”一字，再循环往复。（这一视角后文讲 tokenization 时还会再提。）

大语言模型做的事情就是这么简单，而把模型尺寸与训练数据同步扩大，模型的性能便会随规模可预测地提升，此所谓 Scaling Law；而在这海量数据的学习之中，模型更会慢慢获得种种神奇的能力。


在这个教程中，我们通过训练一个“麻雀虽小，五脏俱全”的模型来学习 LLM 训练和推理的关键技术。

# 1. 数据准备

## 1.1. 下载 Shakespeare 数据集

我们用 Shakespeare 的作品当训练数据，让模型模仿 Shakespeare 的风格写作。

In [ ]:
if not os.path.exists("./data/shakespeare.txt"):
    os.makedirs("./data", exist_ok=True)  # wget -O 不会自动建父目录，先建好 ./data
    !wget -O ./data/shakespeare.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
    #数据集下载到./data，名为shakespeare.txt
else:
    print("file has been downloaded")

## 1.2. 检查数据集

In [ ]:
with open("./data/shakespeare.txt", "r", encoding = "utf-8") as f:
    text = f.read()

# 数据集字符数
print("text len: {}".format(len(text)))

In [ ]:
# 查看数据集前500字符
print(text[:500])

## 1.3. Tokenization

***将文本转换为最小意义单位***

对人而言，Shakespeare 的作品，或者任何的英文文本在语义上的层次结构都是这样的：由 26 个字母组成单词，单词加上标点形成句子，句子组合起来形成段落，段落拼接构成文章。我们看到，每个语义上的层次都有其对应的最小意义单位，从底层往上开始依次是：字母 $\rightarrow$ 单词 $\rightarrow$ 句子 $\rightarrow$ 段落 $\rightarrow$ 文章。然而模型比人要蠢，它不能自动地构建这样的层次结构，也不知道什么是最小意义单位。所以，我们要人为地给模型这样一个先验，即最语义上最底层次的最小意义单位，唤做 token，然后让模型通过对大量数据的学习自发地构建对文本更高层次、更抽象的理解，乃至于融会贯通，产生”涌现“。

对英文文本而言，这 token 不一定要从字符(character)开始，token 可以是字符(character)，可以是字符组合(subword, character sequence)，可以是单词(word)，或者是词组(combination of words)。不同的 token 选取方式各有优劣：用字符作 token 的话，”词表“长度很短（英文文本里不同的字符不出百种），但是从简单的字符到复杂的语义表达之间跨度太大，这样导致模型学习难度巨大；而另一极端假设用词组作 token，单 token 就可表达复杂语义，但这样词表长度特别之长（有太多种不同的词语组合），模型学习的难度同样可能很大。所以，tokenization 是一个需要 trade off 的操作。有很多现成的确定 token 的算法，比如一种从大量文本中按出现频率高低抽取字符组合(subword, character sequence)的算法[BPE](https://github.com/openai/tiktoken)。

大语言模型归根结底是在做这样一件事情：Next token prediction，即输入一串 tokens，模型预测下一个 token 是什么。即：

$$
p(t_{k} | (t_{1}, t_{2}, ..., t_{k-1}), \theta)
$$

给定前 $k-1$ 个 token，模型 $\theta$ 给出第 $k$ 个 token 的概率。而表面上这是一个概率统计模型，而当我们把人类自古以来所有文本（也就是所有知识）都喂给模型训练之后，模型只有理解了这前 $k-1$ 个 token 无数种组合所蕴含的大千世界纷繁复杂之过程，才能合理给出第 $k$ 个 token。

### 1.3.1. 字符作为 token

这里我们首先采用最简单直接方法，将字符作为 token。假设 Shakespeare 的文本中共有 N 个不一样的字符（比如英文字符或者标点符号），我们将每个字符对应一个离散整数（0～N-1，共 N 个）。则 Next token prediction 就变成了这样一个任务：输入一串 tokens（用对应的 0～N-1 之间的一串整数表征），输出一个长度为 N 的向量，向量中的第 i 维代表待预测的 token 为第 i 个字符的概率。

In [ ]:
# 打印所有的字符
chars = sorted(list(set(text)))
print("There are {} chars, which are {}".format(len(chars), chars))

In [ ]:
# 每一个字符对应一个离散的整数
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# 测试 encode 和 decode
print(encode("hi there"))
print(decode(encode("hi there")))

### 1.3.2. (optional)BPE 算法

我们也可以使用更高级的[BPE](https://github.com/openai/tiktoken)（Byte Pair Encoding，字节对编码）算法做 Tokenization。它正是上文提到的那类“从大量文本中按出现频率抽取字符组合(subword)”的算法。

**一句话原理**：从最基础的单位（字符/字节）出发，不断地把“当前语料里出现最频繁的相邻 token 对”合并成一个新 token，如此反复，直到词表增长到预设大小。这样一来，高频的字符组合（常见词、词根、词缀，如 `ing`、`tion`）会被逐步合并成单个 token，而罕见的串则留作较小的碎片——于是在“词表大小”与“序列长度”两端取得折中。这与上文的 trade off 一脉相承：token 粒度越细，词表越小，但序列越长、从字符到语义的跨度也越大，模型越难学；粒度越粗则反之。

**算法流程**：记基础符号集（全部字符或字节）为 $V_0$，并把语料中的每个词都写成基础符号的序列。随后重复以下两步 $M$ 次（$M$ 为预设的合并次数，是一个超参数）：

$$
(x^{*}, y^{*}) = \arg\max_{(x, y)}\ \mathrm{freq}(x, y)
$$

$$
V_{t+1} = V_t \cup \{x^{*} y^{*}\}, \qquad t = 0, 1, \dots, M-1
$$

即：在**当前语料**上统计所有相邻 token 对 $(x, y)$ 的出现频次 $\mathrm{freq}(x, y)$，取频次最高的一对 $(x^{*}, y^{*})$ 合并为新 token $x^{*}y^{*}$ 加入词表，并把语料中的该相邻对**全部**替换成这个新 token（若频次并列，按某种约定取其一）；每合并一次语料就被改写一次，故下一轮的频次需在**新语料**上重新统计。最终词表大小为 $|V| = |V_0| + M$。编码新文本时，只需按学到的合并规则**及其先后顺序**贪心地逐步合并；解码时把各 subword 首尾拼接即可还原。

**一个小例子**：设语料中有如下词及词频 `{hug:10, pug:5, pun:12, bun:4, hugs:5}`，基础词表为 `{b, g, h, n, p, s, u}`。每轮取当前频次最高的相邻对做合并：

| 轮次 | 最高频相邻对 | 频次 | 新增 token |
| :--: | :--: | :--: | :--: |
| 1 | (u, g) | 20 | `ug` |
| 2 | (u, n) | 16 | `un` |
| 3 | (h, ug) | 15 | `hug` |

于是 “hug” 被编码成单个 token `hug`，“hugs” 则是 `hug`+`s`；而一个没在语料里出现过的新词 “bug”，也能借助已学到的规则被切成 `b`+`ug` 两个 token——这正是 BPE 能从容应对未登录词 (OOV) 的原因。

**现代实践**：GPT 系列用的是 **byte-level BPE**，即把合并建立在原始**字节**（而非字符）之上，基础词表恒为 256 个字节，因此任何文本（各种语言、emoji 乃至乱码）都不会 OOV，且因空格、换行乃至任意字符本身都是字节基础符号，可逐字节拼回原文、**完全无损**。下面要用到的 `tiktoken`（`cl100k_base` / `o200k_base`）正是这类字节级 BPE 的高效实现。

更详尽的推导可参考[Introduction to Natural Language Processing: Byte Pair Encoding (BPE) and Natural Language Toolkit (NLTK)](https://medium.com/@RobuRishabh/introduction-to-natural-language-processing-byte-pair-encoding-bpe-and-natural-language-toolkit-414c83f85b71#:~:text=Byte%20Pair%20Encoding%20(BPE)%20is,smaller%20units%20called%20subword%20tokens.&text=Handling%20Out%2Dof%2DVocabulary%20Words,words%20not%20seen%20during%20training.)。


In [ ]:
!pip3 install -U tiktoken

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")#("o200k_base")
assert enc.decode(enc.encode("hello world")) == "hello world"

详细用法请见[OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb)

#### 1.3.2.1. 深入一点：GPT 的 byte-level BPE 具体怎么做

朴素 BPE 直接在**字符**上合并，会遇到两个麻烦：基础词表要囊括所有可能的字符（Unicode 有十几万个码点），而且一旦遇到训练时没见过的字符（生僻字、新 emoji）仍会 OOV。GPT-2 起改在**字节**上做 BPE，干净利落地绕开这两点。具体分四步：

1. **预分词（pre-tokenize）**：先用一个正则把文本切成“词块”（大致按空格 / 标点 / 数字切，并把前导空格并进词里，所以 `" the"` 和 `"the"` 是不同的块）。**BPE 的合并只在词块内部进行、绝不跨块**，这样空格、标点天然成为边界，避免把 `"dog."` 和 `"dog"` 学成同一个 token。（cl100k / o200k 用的是更新版正则，对缩写、连续数字等处理更细。）
2. **转成字节（UTF-8）**：把每个词块按 UTF-8 编码成**字节流**。于是无论什么语言、符号、emoji，都变成 $0\sim255$ 之间的一串字节——**基础词表恒为这 256 个字节**，再不会有“没见过的字符”。
3. **在字节上跑 BPE 合并**：对字节序列执行前面讲的那套“按频率合并最高频相邻对”的流程，学到一张合并表。最终词表 = 256 个基础字节 + 学到的若干合并（cl100k_base 共约 10 万）。注意：常见英文词可能被合成单个 token，而生僻 emoji 可能老老实实拆成好几个字节 token——**代价是 token 变多，但永远不会失败**。
4. **编码 / 解码**：编码时，对每个词块按合并表的优先级贪心合并成 token id；解码时，把每个 token id 还原成它对应的**字节序列**、全部拼起来，再 UTF-8 解码回文本。因为一切最终都落在字节上、且 256 个字节全在词表里，所以**往返完全无损**，emoji、多字节汉字也能逐字节拼回。

> 小细节：GPT-2 的分词器还把 256 个字节一一映射成可打印的 Unicode 字符（可逆），纯粹是为了让词表文件好读、避免空白 / 控制字符捣乱——这就是你在 GPT-2 词表里看到 `Ġthe`（其中 `Ġ` 代表空格）这种写法的原因。`tiktoken`（cl100k）则直接在原始字节上工作，`enc._mergeable_ranks` 存的就是 `字节 -> rank` 的映射。

In [ ]:
# byte-level BPE 直观演示：含重音符、中文、emoji 的字符串
s = "héllo 世界 🤖"
ids = enc.encode(s)
print("原文     :", repr(s))
print("token ids:", ids)

# 每个 token 本质上对应一串“字节”(bytes)
print("逐 token 的字节:")
for i in ids:
    print("   {:>6}  ->  {}".format(i, enc.decode_single_token_bytes(i)))

# 一切落在字节上，故往返完全无损（emoji、多字节汉字也不例外）
print("无损还原 :", enc.decode(ids) == s)

# 基础词表就是 256 个字节 —— 这正是“任何文本都不会 OOV”的根基
base_bytes = sum(1 for b in range(256) if bytes([b]) in enc._mergeable_ranks)
print("词表里的单字节数:", base_bytes, "（= 全部 256 个字节）")
print("cl100k_base 总词表大小 n_vocab:", enc.n_vocab)

In [ ]:
# ==========
use_bpe = False # 默认走字符级（下文“65 个字符 / N=65 / 65 个概率值”均以此为准）；
               # 改成 True 则改用 BPE —— 但注意 vocab_size 会从 65 跳到约 10 万，前文关于 65 的说明不再适用，
               # 且在本玩具规模（n_embed=32、block_size=16）上词表远大于数据，训练与生成效果都会明显变差
# ==========

if use_bpe:
    encode = enc.encode
    decode = enc.decode

    print(encode("hi there"))
    print(decode(encode("hi there")))
    
    vocab_size_bpe = len(enc._mergeable_ranks)

## 1.4. 将 Shakespeare 数据集转化为 tokens

In [ ]:
# 将上述的Shakespeare数据集encode
data = encode(text)
len_data = len(data)
print("len of data: {}".format(len_data))

划分训练集和测试集

In [ ]:
split_point = int(0.9 * len(data))
train_set = data[:split_point]
test_set = data[split_point:]
print(f"len of train_set: {len(train_set)}")
print(f"len of test_set: {len(test_set)}")

# 2. 模型的 hyperparameters 

In [ ]:
# 这些值是我随手设置的，并非最优，可修改

batch_size = 128 # 训练的batch size
block_size = 16 # 输入模型的token的最大数量
if use_bpe:
    vocab_size = vocab_size_bpe
else:
    vocab_size = len(chars) # token词表的长度，which is the number of unique characters in this case
n_embed = 32 # embedding层的维度。表征token的整数输入模型，模型将此整数映射到维度为n_embed的向量。
n_layers = 16 # 模型multihead attention block的层数
head_size = 16 # 模型multihead attention block中每个head的维度
num_heads = 8 # 模型multihead attention block中head的数量（注意 head_size*num_heads=128 并不等于 n_embed=32：q/k/v 先投影到 128 维做注意力、再由 fc 投影回 n_embed，二者在本实现里解耦，这与常见的 d_model=num_heads*head_size 约定不同）

learning_rate = 1e-4 # 学习率
print_interval = 200 # 打印训练信息的间隔，每间隔print_interval个步数打印训练信息
num_epoch = 10 # 训练的epoch数

# 3. 模型的 datasetloader

定义每步训练的输入和输出

e.g.（假设把字符作为 token）

- 模型输入：`e?\n\nDUKE OF AUME` 预测目标：`?\n\nDUKE OF AUMER`

- 模型输入：`rike?\n\nSecond Mu` 预测目标：`ike?\n\nSecond Mur`

In [ ]:
# 定义Dataset和Datasetloader
class MyDataset(Dataset):
    def __init__(self, textdata, block_size):
        self.textdata = textdata
        self.block_size = block_size
    
    def __len__(self):
        return len(self.textdata) - self.block_size  # 需保证 idx+1+block_size <= len，故最后一个合法 idx 为 len-block_size-1，样本数为 len-block_size
    
    def __getitem__(self, idx):
        return {
            "input": torch.tensor(self.textdata[idx:idx+self.block_size], dtype=torch.long),
            "target": torch.tensor(self.textdata[idx+1:idx+1+self.block_size], dtype=torch.long),
        }

train_dataset = MyDataset(train_set, block_size)
train_datasetloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers = 4)

test_dataset = MyDataset(test_set, block_size)
test_datasetloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers = 4)

In [ ]:
# 得到一组输入和预测目标
example_batch = next(iter(train_datasetloader))

# 模型输入和预测目标的tensor shape = (batch_size, block_size)
print("input shape (batch_size, block_size): {}\ntarget shape (batch_size, block_size): {}\n".format(example_batch["input"].shape, example_batch["target"].shape))

# 模型输入和预测目标的示例
print("decoded\nexample input: {}\nexample target: {}\n".format(repr(decode(list(example_batch["input"][0].numpy()))), repr(decode(list(example_batch["target"][0].numpy())))))

# 没有decode的样子
print("without decoding\nexample input: {}\nexample target: {}".format(list(example_batch["input"][0].numpy()), list(example_batch["target"][0].numpy())))

In [ ]:
# 测试datasetloader是否正确

# for i, batch in enumerate(train_datasetloader):
#     if i % 100 == 0:
#         print("{} | {}".format(i, len(train_datasetloader)), end="\r")
# print("{} | {}".format(i, len(train_datasetloader)))
# print("there is no error")

# 4. Embedding

***将 token 映射为高维向量表征语义（个体）与位置关系（相互）***

## 4.1. 语义的 embedding

在 Tokenization 一步中，若我们将 token 定义为字符，且用 0～N-1 的整数表征 token（Shakespeare 数据集中有 65 个不一样的字符，则 N 为 65）。为了让模型能够理解并处理这些离散的整数，我们将每个整数映射为一个长度为 `n_embed` 的向量，向量中的每个维度均为连续值。具体来说，我们这里有 65 个字符，`n_embed`=32，则我们会有一个 $E \in \mathbb{R}^{65 \times 32}$，$E$ 的第 $i$ 行表征第 $i$ 个 token。这样做的意义是，第 $i$ 个 token 对应的长度为 `n_embed` 的向量可以视为模型对第 $i$ 个 token 的抽象理解，且此向量可导，能训练更新。

In [ ]:
# 定义一个仅有一层embedding的模型
class EmbeddingTest(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embed,
    ):
        super().__init__()
        self.token_embed_tab = nn.Embedding(vocab_size, n_embed)

    def forward(self, x):
        B, T = x.shape #B: batch size; T: num of tokens; int of shape (B, T)
        return self.token_embed_tab(x)

embed_test_model = EmbeddingTest(
    vocab_size,
    n_embed,
)

In [ ]:
# 得到一组输入和预测目标
example_batch = next(iter(train_datasetloader))

# 模型输入和预测目标的tensor shape = (batch_size, block_size)
print("input shape (batch_size, block_size): {}\ntarget shape (batch_size, block_size): {}\n".format(example_batch["input"].shape, example_batch["target"].shape))

# 模型输入和预测目标的示例
print("decoded\nexample input: {}\nexample target: {}\n".format(repr(decode(list(example_batch["input"][0].numpy()))), repr(decode(list(example_batch["target"][0].numpy())))))

# 没有decode的样子
print("without decoding\nexample input: {}\nexample target: {}".format(list(example_batch["input"][0].numpy()), list(example_batch["target"][0].numpy())))

In [ ]:
# embedding
embed = embed_test_model(example_batch["input"])
print("shape of embed (batch_size, block_size, n_embed) = {}".format(embed.shape))

## 4.2. Position Embedding

***为 token 添加位置信息***

以上 Embedding 方式有助于让模型理解 token 本身的含义（其特征被模型抽象为一个长度为 `n_embed` 的向量），但是在这样情况下模型无法学习 token“相互”的关系，因为这样的 Embedding 没有包含任何表征“位置”的信息。于是我们引入位置编码（Postion Embedding）。

位置编码方式多种多样，大致可以分为两种：**绝对位置编码**和**相对位置编码**，对于各种编码方式的介绍可以参考[让研究人员绞尽脑汁的Transformer位置编码](https://spaces.ac.cn/archives/8130)。
这里我们展示三种最具代表性的位置编码。

### 4.2.1. 绝对位置-自学习 Position Embedding

最简单直接的一种编码方式：与 4.1 的语义 embedding 一脉相承——再建一张**可学习**的位置表 $P \in \mathbb{R}^{16 \times 32}$（$16$=block_size，$32$=n_embed），第 $i$ 行就是第 $i$ 个位置的编码（随机初始化、随训练更新）。前向时按下标 $0, 1, \dots, T-1$ 查表取出对应行，加到 token embedding 上。这里的 $i$ 只是“第几号位置”的索引，而非被直接当作参数去学习的那个数值。

In [ ]:
# 添加简单的自学习position embedding
class EmbeddingTestWithSelfLearningPosEmbed(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embed,
        block_size
    ):
        super().__init__()
        self.token_embed_tab = nn.Embedding(vocab_size, n_embed)
        self.pos_embed_tab = nn.Embedding(block_size, n_embed)

    def forward(self, x):
        B, T = x.shape #B: batch size; T: num of tokens; int of shape (B, T)
        tok_embed = self.token_embed_tab(x)
        pos_embed = self.pos_embed_tab(torch.arange(T, device = x.get_device() if x.get_device() >= 0 else "cpu"))
        x = tok_embed + pos_embed
        return x, tok_embed, pos_embed

embed_test_model_w_pos = EmbeddingTestWithSelfLearningPosEmbed(
    vocab_size,
    n_embed,
    block_size
)

In [ ]:
embed, tok_embed, pos_embed = embed_test_model_w_pos(example_batch["input"])

# 文本
input_text_list = [decode(list(example_batch["input"].numpy()[i])) for i in range(example_batch["input"].numpy().shape[0])]
target_text_list = [decode(list(example_batch["target"].numpy()[i])) for i in range(example_batch["target"].numpy().shape[0])]

# 模型输入和预测目标的示例
print("decoded\nexample input: {}\nexample target: {}\n".format(repr(input_text_list[0]), repr(target_text_list[0])))

# 尺寸
print("shape of embed (batch_size, block_size, n_embed) = {}".format(embed.shape))
print("shape of tok_embed (batch_size, block_size, n_embed) = {}".format(tok_embed.shape))
print("shape of pos_embed (block_size, n_embed) = {}".format(pos_embed.shape))

### 4.2.2. 绝对位置-Sinusoidal Position Embedding

出自[Attention Is All You Need](https://arxiv.org/abs/1706.03762)

与自学习位置编码不同，Sinusoidal 位置编码是**固定的、无需学习**的：用一组不同频率的正、余弦函数，为每个绝对位置 $pos$ 直接算出一个长度为 $d$（即 `n_embed`）的向量，再加到对应 token 的 embedding 上。

设 $pos \in \{0, 1, \dots, L-1\}$ 为 token 在序列中的位置（$L$ 即 `block_size`），$d$ 为 embedding 维度，$i \in \{0, 1, \dots, d/2-1\}$ 为维度对的编号，则：

$$
\begin{aligned}
PE_{(pos,\, 2i)}   &= \sin\left(\frac{pos}{10000^{2i/d}}\right) \\
PE_{(pos,\, 2i+1)} &= \cos\left(\frac{pos}{10000^{2i/d}}\right)
\end{aligned}
$$

即把 $d$ 个维度两两分组，第 $i$ 组共用一个频率 $\omega_i = 1/10000^{2i/d}$：偶数维（第 $2i$ 维）取 $\sin(\omega_i\, pos)$，奇数维（第 $2i+1$ 维）取 $\cos(\omega_i\, pos)$——这与下方代码的 `pe[:, 0::2] = sin`、`pe[:, 1::2] = cos` 一一对应。随后 `x = tok_embed + pos_embed`。

**直觉**：

- **多尺度的"刻度盘"**：频率 $\omega_i$ 随 $i$ 增大而几何式衰减，波长从 $2\pi$ 一路拉长到约 $2\pi \cdot 10000$。低维像快针、高维像慢针，不同维度以不同速度振荡，组合起来便给每个位置一个几乎唯一的"指纹"（类似用一排不同转速的指针来表示一个整数）。
- **暗含相对位置**：正、余弦成对出现并非偶然——对任意固定偏移 $k$，$PE_{pos+k}$ 都能写成 $PE_{pos}$ 的一个**线性变换**（本质是分块旋转：第 $i$ 组的两维各自转过 $\omega_i k$，各组转角都只随 $k$ 线性变化、与 $pos$ 无关）。于是模型只要学会这个线性关系，就能从绝对编码里"读出"token 间的相对距离。
- **可外推**：因为是确定性公式而非查表学习，理论上能直接算出训练时未见过的更长位置的编码（尽管实际外推效果有限）。

另外可以参考这篇文章[Transformer升级之路：1、Sinusoidal位置编码追根溯源](https://spaces.ac.cn/archives/8231)

In [ ]:
class EmbeddingTestWithSinPosEmbed(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embed,
        block_size,
        base=10000
    ):
        super().__init__()
        self.token_embed_tab = nn.Embedding(vocab_size, n_embed)
        
        self.max_sequence_length = block_size
        self.d_model = n_embed
        self.base = base
        self.register_buffer("pos_embed", self.get_pos_embed())  # 登记为 buffer，.to(device) 时会一起搬走

    def forward(self, x):
        B, T = x.shape #B: batch size; T: num of tokens; int of shape (B, T)
        tok_embed = self.token_embed_tab(x)
        x = tok_embed + self.pos_embed
        return x, tok_embed, self.pos_embed
        
    def get_pos_embed(self):
        pe = torch.zeros(self.max_sequence_length, self.d_model, dtype=torch.float)  # size: (max_sequence_length, d_model) = (block_size, n_embed)，行数是“位置个数”block_size，别与 token embedding 的 vocab_size 混淆
        exp_value = 2*torch.arange(self.d_model // 2, dtype=torch.float) / self.d_model
        alpha = 1 / (self.base ** exp_value)  # size: (d_model/2)
        out = torch.arange(self.max_sequence_length, dtype=torch.float)[:, None] @ alpha[None, :]  # size: (max_sequence_length, d_model/2)
        embedding_sin = torch.sin(out)
        embedding_cos = torch.cos(out)
        pe[:, 0::2] = embedding_sin  # 偶数维(2i)设为sin
        pe[:, 1::2] = embedding_cos  # 奇数维(2i+1)设为cos
        return pe

embed_test_model_w_pos = EmbeddingTestWithSinPosEmbed(
    vocab_size,
    n_embed,
    block_size
)

In [ ]:
embed, tok_embed, pos_embed = embed_test_model_w_pos(example_batch["input"])

# 文本
input_text_list = [decode(list(example_batch["input"].numpy()[i])) for i in range(example_batch["input"].numpy().shape[0])]
target_text_list = [decode(list(example_batch["target"].numpy()[i])) for i in range(example_batch["target"].numpy().shape[0])]

# 模型输入和预测目标的示例
print("decoded\nexample input: {}\nexample target: {}\n".format(repr(input_text_list[0]), repr(target_text_list[0])))

# 尺寸
print("shape of embed (batch_size, block_size, n_embed) = {}".format(embed.shape))
print("shape of tok_embed (batch_size, block_size, n_embed) = {}".format(tok_embed.shape))
print("shape of pos_embed (block_size, n_embed) = {}".format(pos_embed.shape))

### 4.2.3. 相对位置-RoPE

RoPE（Rotary Position Embedding，旋转位置编码）换了一条思路：不再把位置信息**加**到 embedding 上，而是按位置把向量**旋转**一个角度；并且它作用在进入注意力的 query / key 上（本节演示代码为直观起见，直接作用在 token embedding 上）。

把 $d$ 维向量两两分组成 $d/2$ 个二维平面，第 $i$ 组配一个频率 $\theta_i = 1/10000^{2i/d}$（$i = 0, \dots, d/2-1$，与 Sinusoidal 同款的几何衰减）。位于位置 $m$ 的向量 $x$，其第 $i$ 组分量 $(x_{2i},\, x_{2i+1})$ 被旋转角度 $m\theta_i$：

$$
\begin{pmatrix} x'_{2i} \\ x'_{2i+1} \end{pmatrix}
=
\begin{pmatrix} \cos m\theta_i & -\sin m\theta_i \\ \sin m\theta_i & \cos m\theta_i \end{pmatrix}
\begin{pmatrix} x_{2i} \\ x_{2i+1} \end{pmatrix}
$$

写成复数更紧凑（也正是代码中 `view_as_complex` + `torch.polar` 的做法）：令 $z_i = x_{2i} + \mathrm{i}\, x_{2i+1}$，旋转即乘上一个单位复数：

$$
\tilde{z}_i = z_i \cdot e^{\mathrm{i}\, m \theta_i}
$$

**intuition（为何"绝对地旋转"却能表达"相对位置"）**：

记 $R_m$ 为"按位置 $m$ 做上述分块旋转"这一操作。注意力的核心是 query 与 key 的内积；把 RoPE 分别作用在位置 $m$ 的 $q$ 和位置 $n$ 的 $k$ 上，二者内积为

$$
\langle R_m\, q,\ R_n\, k \rangle = q^{\top} R_m^{\top} R_n\, k = q^{\top} R_{n-m}\, k
$$

因为旋转矩阵满足 $R_m^{\top} R_n = R_{n-m}$。换言之：**虽然施加给每个 token 的是"绝对位置"的旋转，但注意力真正感受到的只是两者的相对距离 $n-m$**。这正是 RoPE 的精妙——以绝对位置的方式实现（简单、且无需额外的两两位置偏置矩阵），却收获相对位置的效果。

一个形象的比喻：把每个 token 的内容看作平面上的一支箭，位置越靠后、箭就被多转一些；两支箭各自被转过的角度之差（即 $(n-m)\theta_i$）只取决于它们相隔多远，而与各自的绝对位置无关；不同维度组以不同转速旋转，用来区分"隔得近"还是"隔得远"。

正是这种“以绝对位置之实、收相对位置之效”的巧劲，加上实现简单、与注意力机制天然契合，使 RoPE 被 Llama 3、Qwen、DeepSeek、Gemma、Mistral 等几乎所有现代开源大模型采用。不过需留意：原始 RoPE 直接外推到训练长度之外时效果往往有限，实践中常配合 NTK-aware、位置插值（Position Interpolation）/ YaRN 等技巧来扩展上下文长度——这也正是下方所引博客系列中多篇专门探讨的话题。

详见[Transformer升级之路：2、博采众长的旋转式位置编码](https://spaces.ac.cn/archives/8265/comment-page-1)

In [ ]:
class EmbeddingTestWithRoPE(nn.Module):
    def __init__(
        self,
        n_embed,
        block_size,
        base=10000
    ):
        super().__init__()        
        self.max_sequence_length = block_size
        self.d_model = n_embed
        self.base = base
        self.register_buffer("freqs_cis", self.init_pos_embed())  # 登记为 buffer，.to(device) 时会一起搬走

    def forward(self, x):
        batch_size, sequence_length, d_model = x.shape
        assert self.d_model == d_model and self.max_sequence_length == sequence_length
        
        h = x.float().reshape(*x.shape[:-1], -1, 2) # --> shape: (batch_size, self.max_sequence_length, self.d_model//2, 2)
        h = torch.view_as_complex(h) # --> shape: (batch_size, self.max_sequence_length, self.d_model//2)
        
        # 旋转操作
        # 设h中的batch维度的第b个sample中的第t个token的第d维度，即(b, t, d)，为 m + ni，则将其与freqs_cis中对应的虚数相乘，即与(t, d)位置虚数，假设旋转角度为a，则 (m + ni) * (cos(a)+sin(a)i)
        # h与freqs_cis对应位置虚数相乘之后转化为实数，则shape: (batch_size, self.max_sequence_length, self.d_model//2) --> (batch_size, self.max_sequence_length, self.d_model)
        h = torch.view_as_real(h * self.freqs_cis).flatten(2) # --> shape: (batch_size, self.max_sequence_length, self.d_model)
        h = h.type_as(x)
        
        return h, x
        
    def init_pos_embed(self):
        # 将dim维度元素两两分组之后，计算每组每个dim元素的旋转角度
        freqs = 1.0 / (self.base ** (torch.arange(0, self.d_model, 2)[:(self.d_model//2)].float() / self.d_model)) # shape: (self.d_model//2)
        # 生成token序列纬度的索引
        t = torch.arange(self.max_sequence_length, device=freqs.device) # shape: (self.max_sequence_length)
        # freqs与t的外积，(i,j)为第i个token的第j维的旋转角度
        freqs = torch.outer(t, freqs).float() # shape: (self.max_sequence_length, self.d_model//2)
        # 将freqs中的旋转角度转换为复数格式，freqs的(i,j)位置旋转角度为a，则转换后freqs_cis的(i,j)位置变化为cos(a)+sin(a)i
        freqs_cis = torch.polar(torch.ones_like(freqs), freqs) # shape: (self.max_sequence_length, self.d_model//2)
        return freqs_cis

embed_test_model_rope = EmbeddingTestWithRoPE(
    n_embed,
    block_size
)

In [ ]:
_, tok_embed, _ = embed_test_model_w_pos(example_batch["input"])

# 文本
input_text_list = [decode(list(example_batch["input"].numpy()[i])) for i in range(example_batch["input"].numpy().shape[0])]
target_text_list = [decode(list(example_batch["target"].numpy()[i])) for i in range(example_batch["target"].numpy().shape[0])]

# 模型输入和预测目标的示例
print("decoded\nexample input: {}\nexample target: {}\n".format(repr(input_text_list[0]), repr(target_text_list[0])))

# token embed
print("shape of tok_embed (batch_size, block_size, n_embed) = {}".format(tok_embed.shape))

# 添加RoPE
x_add_rope, _ = embed_test_model_rope(tok_embed)
print("shape of x (batch_size, block_size, n_embed) adding rope = {}".format(x_add_rope.shape))

# 5. Attention in Transformer

很多文章都详细解释过 Transformer 的 Attention 原理，推荐下面这两篇文章

- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)（”图画得很好“）

- [Attention? Attention!](https://lilianweng.github.io/posts/2018-06-24-attention/)（OpenAI 大佬写的，且回顾了 Attention 的历史，在 Transformer 之前有很多不同版本的 Attention 方式）

不过这里还是从代码角度解释一下 Transformer 中的 Attention

首先，为什么需要 Attention？语言模型的 Next token prediction 归根结底是在做这样一件事情：

$$
p(t_{k} | (t_{1}, t_{2}, ..., t_{k-1}), \theta)
$$

在模型参数为 $\theta$ 的条件下，给定 tokens: $t_{1}, t_{2}, ..., t_{k-1}$，预测下一个 token 为 $t_{k}$ 的概率（在上述 Shakespeare 数据集例子中，$t_{k}$ 可以是 65 个字符中的任何一个，我们会得到 65 个概率值，概率值最高表明下一个 token 为相应字符的概率最高）。在这样的框架下，预测下一个字符需要参考之前的字符。那问题是，具体用什么方式参考？以前的 RNN、LSTM 设计了一些“参考”的技巧，Attention 被提出来也是这个目的，不过实践证明 Attention 的效果更好。

讨论 Attention，最关键的是这个公式（相信大家看了无数遍了，我还是摆在这里）：

$$
Attention(Q, K ,V) = softmax(\frac{QK^{T}}{\sqrt{d}})V
$$


现在先不管这个公式在做什么，回到我们的代码中来。在经过 Embedding 处理之后，每个 token 被抽象成一个长度为 `n_embed` 的向量，即，在这个 batch 中训练样本有 `batch_size` 个，在对每个训练样本，输入 token 的数量有 `block_size` 个，而每个 token 由一个长度为 `n_embed` 的向量表征。

In [ ]:
x = embed
print("shape of x (batch_size, block_size, n_embed) = {}".format(x.shape))

# 此处用x代表模型中间层特征，我们对这个x做如下的Attention操作...

## 5.1. Self-Attention

Transformer 的 Attention 操作中的 Key 和 Value 可以和 Query 来源不同（来源不同的版本就成了 Cross-Attention），而三者来源相同时候就是 Self-Attention。我们这里要构造的语言模型用的都是 Self-Attention。

In [ ]:
# 经过Linear层得到Query，Key，Value
class QueryKeyValue(nn.Module):
    def __init__(
        self,
        n_embed,
        head_size,
    ):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias = False)
        self.query = nn.Linear(n_embed, head_size, bias = False)
        self.value = nn.Linear(n_embed, head_size, bias = False)
    
    def forward(self, x):
        return self.query(x), self.key(x), self.value(x)

get_query_key_value = QueryKeyValue(n_embed, head_size=n_embed) # 这里Query，Key，Value的维度(head_size)仍然为n_embed

将输入 `x` 分别过三个 Linear 层之后得到 Query，Key，Value（即三者都是源自于 `x`），分别是 `q, k ,v`

In [ ]:
q, k ,v = get_query_key_value(x)

print("shape of query (batch_size, block_size, head_size): {}".format(q.shape))
print("shape of key (batch_size, block_size, head_size): {}".format(k.shape))
print("shape of value (batch_size, block_size, head_size): {}".format(v.shape))

这里 batch size > 1，我们取其中任一样本做展示，将 batch size 降为 1，与 batch size > 1 的时候同理

In [ ]:
q, k, v = q[:1], k[:1], v[:1] # shape: (1, block_size, head_size)
sample_input_text = input_text_list[0]
sample_target_text = target_text_list[0]
print(repr("Now, the corresponding input text is {} and target text is {}".format(repr(sample_input_text), repr(sample_target_text))))

首先，我们先来做 $\frac{QK^{T}}{\sqrt{d}}$ 这一步，$d$ 为 Q、K 向量的最后一维（本演示中因上一格取了 head_size=n_embed，故 $d$=n_embed=32；真实模型里 $d$=head_size）。

In [ ]:
weights = q @ k.transpose(-2, -1) * q.shape[-1]**-0.5 # 用 Q/K 实际最后一维来缩放（本演示因上面取 head_size=n_embed，故相当于除以 sqrt(n_embed)）
print("shape: {}".format(weights.shape)) # shape: (batchsize, num of tokens, num of tokens)

解释一下为什么需要 $\frac{1}{\sqrt{d}}$？这是为了控制方差。若不除以 $\sqrt{d}$，$QK^{T}$ 的数值会随维度 $d$ 增大而变大，经 softmax 后概率会过度集中到某一个 token（接近 one-hot），使梯度极小、训练难以进行；除以 $\sqrt{d}$ 把方差拉回 $O(1)$，softmax 才不至于饱和。下面用随机数直观看看方差的变化：

In [ ]:
# 使用随机数做示例
k_rand = torch.randn(batch_size, block_size, head_size)
q_rand = torch.randn(batch_size, block_size, head_size)
weights_rand = q_rand @ k_rand.transpose(-2, -1) * head_size**-0.5
weights_rand_without_frac_d = q_rand @ k_rand.transpose(-2, -1)

print("var of k_rand: {}".format(k_rand.var()))
print("var of q_rand: {}".format(q_rand.var()))
print("var of weights_rand: {}".format(weights_rand.var()))
print("when 1/sqrt(d) is not applied, var of weights_rand_without_frac_d: {}".format(weights_rand_without_frac_d.var()))

In [ ]:
# 对于我们的q, k 
print("var of k: {}".format(k.var()))
print("var of q: {}".format(q.var()))
print("var of weights: {}".format(weights.var()))
print("without 1/sqrt(d), var of q @ k.transpose(-2, -1): {}".format((q @ k.transpose(-2, -1)).var()))

那么 $\frac{QK^{T}}{\sqrt{d}}$ 这个式子本身有什么含义呢？

这里 `weights = q @ k.transpose(-2, -1) * head_size**-0.5` 这一步的 shape 变换为 `(1, block_size, head_size) @ (1, head_size, block_size) --> (1, block_size, block_size)`，得到的 `weights` 中第 $i$ 行第 $j$ 列表征第 $j$ 个 token 对第 $i$ 个 token 的重要度

我们把 `weights` 的结果画出来

In [ ]:
attn_tick_labels = [decode([token]) for token in encode(sample_input_text)] #[*sample_input_text]
for i in range(len(attn_tick_labels)):
    if attn_tick_labels[i] == "\n":
        attn_tick_labels[i] = "\\n"
plt.figure(1)
plt.xticks(ticks=range(len(attn_tick_labels)), labels=attn_tick_labels, rotation=90)
plt.yticks(ticks=range(len(attn_tick_labels)), labels=attn_tick_labels)
plt.imshow(weights.detach().numpy()[0])
plt.colorbar()
plt.title("weights")
plt.show()

但是对于 Next token prediction 的任务，这样的 Attention 有个问题，即第 $i$ 个 token 只能看到它自己及之前的 token（e.g., 第 $1, 2, ..., i$ 个 token），但不应该看到之后的 token（下面用的 `tril` 含对角线，正是为了保留每个 token 对“自己”的注意）。对此引入了 Causal Attention 操作，即：

In [ ]:
# shape = (block_size, block_size)的下三角矩阵
tril_matrix = torch.tril(torch.ones(block_size, block_size))
print(tril_matrix)

对于上文得到的 `weights`，将对应 `tril_matrix` 为 0 的位置置为 `-inf`，即对于 `weights` 中第 $i$ 行（表征第 $i$ 个 token），所有 $>i$ 的 token 对第 $i$ 个 token 没有影响（重要度为负无穷）

In [ ]:
weights = weights.masked_fill(tril_matrix == 0, float("-inf"))
plt.figure(1)
plt.xticks(ticks=range(len(attn_tick_labels)), labels=attn_tick_labels, rotation=90)
plt.yticks(ticks=range(len(attn_tick_labels)), labels=attn_tick_labels)
plt.imshow(weights.detach().numpy()[0])
plt.colorbar()
plt.title("causal weights")
plt.show()

再进行 `softmax` 操作，即得到 $softmax(\frac{QK^{T}}{\sqrt{d}})$

In [ ]:
weights = F.softmax(weights, dim=-1)
plt.figure(1)
plt.xticks(ticks=range(len(attn_tick_labels)), labels=attn_tick_labels, rotation=90)
plt.yticks(ticks=range(len(attn_tick_labels)), labels=attn_tick_labels)
plt.imshow(weights.detach().numpy()[0])
plt.colorbar()
plt.title("softmax of causal weights")
plt.show()

于是乎，这里的第 $i$ 行表征了所有 token 对第 $i$ 个 token 的重要性，当然，所有第 $i$ 个之后的 token 的重要性值为 0

In [ ]:
ith = torch.randint(low=0, high=weights.shape[1], size=(1,))[0].item()
wei_val_ith = weights[0, ith].detach().cpu().numpy()
print("the {}th row of weights: {}\n\nthe sum of the weights of the {}th row is {}\n".format(ith+1, wei_val_ith, ith+1, weights[0, ith].sum()))
print("which means, for the {}th token '{}'".format(ith+1, attn_tick_labels[ith]))
for j, (cur_wei_val, label) in enumerate(zip(wei_val_ith, attn_tick_labels)):
    print("the importance of {}th token '{}' to {}th token '{}' is {}".format(j+1, label, ith+1, attn_tick_labels[ith], cur_wei_val))

对于 $Attention(Q, K ,V) = softmax(\frac{QK^{T}}{\sqrt{d}})V$，我们已经得到了 $softmax(\frac{QK^{T}}{\sqrt{d}})$ 并且知道其含义（上文所述的 `weights`，表征 token 之间的重要性程度），那 $softmax(\frac{QK^{T}}{\sqrt{d}})$ 与 $V$ 矩阵相乘是什么含义呢？

如上文所述，$softmax(\frac{QK^{T}}{\sqrt{d}})$ 的第 $i$ 行表征其他所有 token 对第 $i$ 个 token 的重要性程度，$V$ 的第 $i$ 行则表征第 $i$ 个 token 的特征

则 $softmax(\frac{QK^{T}}{\sqrt{d}})$ 与 $V$ 相乘可看作 $softmax(\frac{QK^{T}}{\sqrt{d}})$ 的每一行和 $V$ 相乘

$softmax(\frac{QK^{T}}{\sqrt{d}})$ 的第 $i$ 行与 $V$ 相乘的含义为，对 $V$ 中所有 token 的特征赋予权重，然后相加，得到与其他 token 进行了 attention 操作后的第 $i$ 个 token 的特征，即

$$
softmax(\frac{QK^{T}}{\sqrt{d}})的第i行: softmax(\frac{QK^{T}}{\sqrt{d}})_{i} = [a_{1 \rightarrow i}, a_{2 \rightarrow i}, ..., a_{b \rightarrow i}]
$$

$$
V = [f_{1}, f_{2}, ..., f_{b}]^{T}
$$

$$
softmax(\frac{QK^{T}}{\sqrt{d}})_{i}V = a_{1 \rightarrow i}f_{1} + a_{2 \rightarrow i}f_{2} + a_{b \rightarrow i}f_{b}
$$

where $b$ 为 `block_size`；$a_{j \rightarrow i}$ 是一个 0～1 之间的数，表征第 $j$ 个 token 对第 $i$ 个 token 的重要性程度（如上文所述）；$f_{j} \in \mathbb{R}^{1 \times d}$ 为第 $j$ 个 token 的特征，其中 $d$ 为 `head_size`。

In [ ]:
# 由此得到输出
out = weights @ v # (1, block_size, block_size) @ (1, block_size, head_size) --> (1, block_size, head_size)
print("shape of out: {}".format(out.shape))

综上所述我们将代码整理

In [ ]:
class SimpleSelfAttn(nn.Module):
    def __init__(
        self,
        n_embed,
        head_size,
        block_size_max,
    ):
        super().__init__()
        self.n_embed = n_embed
        self.head_size = head_size
        
        self.key = nn.Linear(n_embed, head_size, bias = False)
        self.query = nn.Linear(n_embed, head_size, bias = False)
        self.value = nn.Linear(n_embed, head_size, bias = False)
        self.fc = nn.Linear(head_size, n_embed) # 将特征的维度从head_size映射回n_embed
        self.register_buffer("mask", torch.tril(torch.ones(block_size_max, block_size_max)))
        
    def forward(self, x):
        # x.shape = (batch_size, block_size, n_embed)
        # attn: softmax(QK^T/sqrt(d))V
        # where Q shape: (batch_size, block_size, head_size), K shape: (batch_size, block_size, head_size), V shape: (batch_size, block_size, head_size)
        batch_size, seq_length, n_embed = x.size()
        assert n_embed == self.n_embed
        q, k, v = self.query(x), self.key(x), self.value(x)
        w = q @ k.transpose(-2,-1) * (self.head_size**-0.5) # (batch_size, block_size, head_size) @ (batch_size, head_size, block_size) --> (batch_size, block_size, block_size)
        w = w.masked_fill(self.mask[:w.shape[-2],:w.shape[-1]]==0, float("-Inf"))
        w = F.softmax(w, dim=-1)
        out = w @ v # (batch_size, block_size, block_size) @ (batch_size, block_size, head_size) --> (batch_size, block_size, head_size)
        out = self.fc(out) # --> (batch_size, block_size, n_embed)
        return out

simple_self_attn = SimpleSelfAttn(
    n_embed = n_embed,
    head_size = head_size,
    block_size_max = block_size,
)

In [ ]:
print("shape of input of attention module: {}".format(x.shape))
print("shape of output of attention module: {}".format(simple_self_attn(x).shape))

## 5.2. Multi-head Attention

在上述 Self-Attention 基础上引入一种名为 Multi-head Attention 技巧。这名字听起来花里胡哨的，本质上是把 `q, k, v` 从单个 head 扩展成多个 head：先把投影加宽到 `head_size * num_heads` 维，再切成 `num_heads` 个各自 `head_size` 维的小 head，即 `(batch_size, block_size, head_size*num_heads) --> (batch_size, block_size, num_heads, head_size)`（注意 `head_size * num_heads` 不必等于 `n_embed`，最后由 `fc` 投影回 `n_embed`）。多出来的 `num_heads` 维让模型能在多个子空间里各学一套注意力特征。另外 Multi-head Attention 也是并行计算的，计算效率高。

In [ ]:
class MultiHeadAttn(nn.Module):
    def __init__(
        self,
        n_embed,
        head_size,
        num_heads, # 定义multi-head attention中head的数量
        block_size_max,
    ):
        super().__init__()
        self.n_embed = n_embed
        self.head_size = head_size
        self.num_heads = num_heads
        
        self.key = nn.Linear(n_embed, head_size*num_heads, bias = False)
        self.query = nn.Linear(n_embed, head_size*num_heads, bias = False)
        self.value = nn.Linear(n_embed, head_size*num_heads, bias = False)
        self.fc = nn.Linear(head_size*num_heads, n_embed) # 将特征的维度从head_size映射回n_embed
        self.register_buffer("mask", torch.tril(torch.ones(block_size_max, block_size_max)))

    # 分head的函数
    # ====================
    def split_heads(self, h):
        batch_size, block_size, total_head_size = h.size()
        assert total_head_size == self.head_size * self.num_heads
        h = h.reshape(batch_size, block_size, self.num_heads, self.head_size)
        h = h.permute(0, 2, 1, 3)
        return h
    # ====================

    def forward(self, x):
        # x.shape = (batch_size, block_size, n_embed)
        
        # attn: softmax(QK^T/sqrt(d))V
        # where Q shape: (batch_size, block_size, head_size*num_heads),
        # K shape: (batch_size, block_size, head_size*num_heads),
        # V shape: (batch_size, block_size, head_size*num_heads)
        
        batch_size, seq_length, n_embed = x.size()
        assert n_embed == self.n_embed
        q, k, v = self.query(x), self.key(x), self.value(x)

        # 将q, k, v分出num_heads这个维度
        # ====================
        q = self.split_heads(q) # (batch_size, block_size, head_size*num_heads) --> (batch_size, num_heads, block_size, head_size)
        k = self.split_heads(k)
        v = self.split_heads(v)
        # ====================
        
        w = q @ k.transpose(-2,-1) * (self.head_size**-0.5) # (batch_size, num_heads, block_size, head_size) @ (batch_size, num_heads, head_size, block_size) --> (batch_size, num_heads, block_size, block_size)
        w = w.masked_fill(self.mask[:w.shape[-2],:w.shape[-1]]==0, float("-Inf"))
        w = F.softmax(w, dim=-1)
        out = w @ v # (batch_size, num_heads, block_size, block_size) @ (batch_size, num_heads, block_size, head_size) --> (batch_size, num_heads, block_size, head_size)

        # 调整out的形状，将不同head的特征融合
        # ====================
        out = out.permute(0,2,1,3) # (batch_size, num_heads, block_size, head_size) --> (batch_size, block_size, num_heads, head_size)
        out = out.reshape(batch_size, seq_length, self.num_heads*self.head_size) # (batch_size, block_size, num_heads, head_size) --> (batch_size, block_size, num_heads*head_size)
        # ====================
        
        out = self.fc(out) # (batch_size, block_size, num_heads*head_size) --> (batch_size, block_size, n_embed)
        return out

multi_head_attn = MultiHeadAttn(
    n_embed = n_embed,
    head_size = head_size,
    num_heads = num_heads,
    block_size_max = block_size,
)

In [ ]:
print("shape of input of multi-head attention module: {}".format(x.shape))
print("shape of output of multi-head attention module: {}".format(multi_head_attn(x).shape))

## 5.3. KV cache：推理为什么能更快

上述的两种 Attention 我们并未区别训练和推理，但是训练只需要“预测下一个 token”，但是推理则需要循环往复地预测——即把预测结果作为条件再预测下一个。如果推理未启用缓存，每生成一个 token 都把**整段**重新喂进模型跑一遍前向——于是前面那些 token 的 key / value 被**反复重算**，白白浪费。

于是乎，有了“空间换时间”的**KV cache**技术，思路很简单：因为是因果(causal)注意力，所以过去 token 的表示永远不会因为后面新增 token 而改变。我们进行一下非常简单的数学推导，目前已有 $2$ 个 token，则 $Q = [q_{1}; q_{2}], K = [k_{1}; k_{2}], V = [v_{1}; v_{2}]$，那我们的计算结果也是两位，第一位可以简写成 $softmax_{row-wise}(q_{1}k_{1}^{T})v_{1}$，只和第一个 token 相关，第二位可以简写成 $softmax_{row-wise}(q_{2}k_{1}^{T})v_{1} + softmax_{row-wise}(q_{2}k_{2}^{T})v_{2}$，只和第一二位 token 相关。这里 $softmax_{row-wise}$ 是一种简写，说的是关于 $softmax$ 的操作，只需要关注和哪个 $q, k$ 相关就行。那我们根据前两个 token 计算下一个 token，结果应为 $softmax_{row-wise}(q_{3}k_{1}^{T})v_{1} + softmax_{row-wise}(q_{3}k_{2}^{T})v_{2} + softmax_{row-wise}(q_{3}k_{3}^{T})v_{3}$。可以看到 $k_{1}, v_{1}, k_{2}, v_{2}$ 还是原来的，只需要新算 $q_{3}, k_{3}, v_{3}$ 即可。

**KV cache**方法就显而易见了，把历史 token 的 $K, V$ **缓存**起来；每步只为**新来的那一个 token** 计算 $q, k, v$，把新的 $k, v$ 追加进缓存，再让新 query 与“缓存里的全部 $K/V$”做一次注意力即可。于是每步的注意力从“对整段 $O(T)$ 个 token 重算”降到“只算 1 个新 token、读一遍缓存”，自回归生成大大加速——代价是显存里要常驻这份缓存，其大小正比于 KV head 的数量。

**为什么只缓存 K、V，不连 Q 一起缓存？** 因为缓存的意义是“存下来以后还要再用”：历史 token 的 key / value 会被之后**每一步**的新 query 反复查用，值得存；而每个 query 只在它自己那一步用一次，注意力矩阵每步只新增**一行**（新 query）、却要用到**所有列**（全部 key / value）：列被反复复用，故缓存；行用过即弃，所以没有 q-cache。

下面用一个单头 causal self-attention 的最小例子，把 KV cache 的机制看清楚，并验证它与“整段重算”逐元素一致：

In [ ]:
# KV cache 原理演示：以单头 causal self-attention 为例，
# 对比“每步把整段序列重算一遍”(use_cache=False 的对照实现) 与 “缓存 K/V、每步只算新 token”(KV cache)
torch.manual_seed(0)
d = head_size
# 随便造一组投影权重
Wq = torch.randn(d, d)
Wk = torch.randn(d, d)
Wv = torch.randn(d, d)

def causal_attn(X):                # 不带缓存：对整段 X 做一次 causal self-attention
    q, k, v = X @ Wq, X @ Wk, X @ Wv
    w = q @ k.transpose(-2, -1) * d ** -0.5
    T = X.shape[0]
    w = w.masked_fill(torch.tril(torch.ones(T, T)) == 0, float("-inf"))  # 因果 mask
    return F.softmax(w, dim=-1) @ v

T = 6
X = torch.randn(T, d)              # 假设这是逐步到来的 T 个 token 的特征

# 做法 A（未启用 KV cache）：每来一个新 token，就把 [0..t] 整段重算，取最后一个位置的输出
out_recompute = [causal_attn(X[:t + 1])[-1] for t in range(T)]

# 做法 B（KV cache）：只为“新来的那一个 token”算 q/k/v，把新 k/v 追加进缓存，
# 再让新 query 和“缓存里的全部 K/V”做注意力（天然因果，连 mask 都不需要）
K_cache, V_cache, out_cache = [], [], []
for t in range(T):
    x_t = X[t]
    q_t, k_t, v_t = x_t @ Wq, x_t @ Wk, x_t @ Wv
    K_cache.append(k_t); V_cache.append(v_t)       # 历史 k/v 不再重算，只追加新的
    K = torch.stack(K_cache); V = torch.stack(V_cache)
    w = (q_t @ K.transpose(-2, -1)) * d ** -0.5
    out_cache.append(F.softmax(w, dim=-1) @ V)

# 两者逐元素相等：KV cache 只是省掉重复计算，结果不变
print("KV cache 与整段重算结果一致:", torch.allclose(torch.stack(out_recompute), torch.stack(out_cache), atol=1e-5))
print("整段重算共做 {} 次 token 前向，KV cache 只做 {} 次".format(sum(range(1, T + 1)), T))

给上述 `MultiHeadAttn` 加上**KV cache**

In [ ]:
class MultiHeadAttnKVCache(nn.Module):
    def __init__(
        self,
        n_embed,
        head_size,
        num_heads, # 定义multi-head attention中head的数量
        block_size_max,
    ):
        super().__init__()
        self.n_embed = n_embed
        self.head_size = head_size
        self.num_heads = num_heads
        
        self.key = nn.Linear(n_embed, head_size*num_heads, bias = False)
        self.query = nn.Linear(n_embed, head_size*num_heads, bias = False)
        self.value = nn.Linear(n_embed, head_size*num_heads, bias = False)
        self.fc = nn.Linear(head_size*num_heads, n_embed) # 将特征的维度从head_size映射回n_embed
        self.register_buffer("mask", torch.tril(torch.ones(block_size_max, block_size_max)))

    # 分head的函数
    # ====================
    def split_heads(self, h):
        batch_size, block_size, total_head_size = h.size()
        assert total_head_size == self.head_size * self.num_heads
        h = h.reshape(batch_size, block_size, self.num_heads, self.head_size)
        h = h.permute(0, 2, 1, 3)
        return h
    # ====================

    def forward(self, x, past_kv = None, use_cache = False):
        # x.shape = (batch_size, block_size, n_embed)
        
        # attn: softmax(QK^T/sqrt(d))V
        # where Q shape: (batch_size, block_size, head_size*num_heads),
        # K shape: (batch_size, block_size, head_size*num_heads),
        # V shape: (batch_size, block_size, head_size*num_heads)
        
        batch_size, seq_length, n_embed = x.size()
        assert n_embed == self.n_embed
        q, k, v = self.query(x), self.key(x), self.value(x)

        # 将q, k, v分出num_heads这个维度
        # ====================
        q = self.split_heads(q) # (batch_size, block_size, head_size*num_heads) --> (batch_size, num_heads, block_size, head_size)
        k_new = self.split_heads(k)
        v_new = self.split_heads(v)
        # ====================
        
        if past_kv is not None:
            past_k, past_v = past_kv
            k = torch.cat([past_k, k_new], dim = 2) # 沿 time 维拼接：历史 + 新 token
            v = torch.cat([past_v, v_new], dim = 2)
        else:
            k, v = k_new, v_new
            
        w = q @ k.transpose(-2,-1) * (self.head_size**-0.5) # (batch_size, num_heads, block_size, head_size) @ (batch_size, num_heads, head_size, block_size) --> (batch_size, num_heads, block_size, block_size)
        # 因果 mask 需按已缓存长度偏移：第 i 个新 query（绝对位置 past_len+i）只能看 key 位置 ≤ past_len+i
        T_kv = k.size(2)
        past_len = T_kv - seq_length  # 已缓存的历史长度（=新 query 的起始绝对位置）
        w = w.masked_fill(self.mask[past_len:past_len + seq_length, :T_kv] == 0, float("-Inf"))
        w = F.softmax(w, dim=-1)
        out = w @ v # (batch_size, num_heads, block_size, block_size) @ (batch_size, num_heads, block_size, head_size) --> (batch_size, num_heads, block_size, head_size)

        # 调整out的形状，将不同head的特征融合
        # ====================
        out = out.permute(0,2,1,3) # (batch_size, num_heads, block_size, head_size) --> (batch_size, block_size, num_heads, head_size)
        out = out.reshape(batch_size, seq_length, self.num_heads*self.head_size) # (batch_size, block_size, num_heads, head_size) --> (batch_size, block_size, num_heads*head_size)
        # ====================
        
        out = self.fc(out) # (batch_size, block_size, num_heads*head_size) --> (batch_size, block_size, n_embed)
        return (out, (k,v)) if use_cache else out

multi_head_attn_kv_cache = MultiHeadAttnKVCache(
    n_embed = n_embed,
    head_size = head_size,
    num_heads = num_heads,
    block_size_max = block_size,
)

In [ ]:
# 验证 MultiHeadAttnKVCache：逐 token 增量解码(带 KV cache) 的结果，与“整段重算”逐元素一致
torch.manual_seed(0)
T = 10
seq = torch.randn(1, T, n_embed)   # 假设逐步到来的 T 个 token 的特征

with torch.no_grad():
    full = multi_head_attn_kv_cache(seq)                              # 做法 A：整段一次性前向（参考）
    past, steps = None, []                                           # 做法 B：逐 token 增量解码
    for t in range(T):
        out_t, past = multi_head_attn_kv_cache(seq[:, t:t + 1, :], past_kv=past, use_cache=True)
        steps.append(out_t)
    incr = torch.cat(steps, dim=1)

ok = torch.allclose(full, incr, atol=1e-5)
print("增量解码 == 整段重算:", ok, "| 最大误差 {:.1e}".format((full - incr).abs().max().item()))
assert ok, "KV-cache 增量解码应与整段重算逐元素一致"
past_k, past_v = past
print("缓存 K/V 形状 (B, num_heads, T, head_size):", tuple(past_k.shape))

## 5.4. Group Query Attention

Multi-head Attention 虽好，但在**自回归推理**时有个痛点：每生成一个新 token，都要拿它的 query 去和前面所有 token 的 key、value 做注意力。为避免重复计算，实践中会把历史 token 的 key、value 缓存下来，这便是 **KV cache**。而 MHA 里每个 query head 都各配一套 key/value head，于是 KV cache 的大小正比于 head 数量——当序列很长、模型很大时，读写这块缓存所需的显存与带宽往往成了推理速度的瓶颈。

**Group Query Attention（GQA，分组查询注意力）** 的想法很直接：让多个 query head **共享**同一套 key/value head。把 $H$ 个 query head 分成 $G$ 组（$G$ 整除 $H$），同一组内的 $H/G$ 个 query head 共用一个 key/value head；这样 key/value head 的数量就从 $H$ 降到 $G$，KV cache 也随之缩小为原来的 $G/H$。

它恰好把两个极端连了起来：

| 方式 | key/value head 数 | 说明 |
| :-- | :--: | :-- |
| **MHA**（Multi-Head Attention） | $G = H$ | 每个 query head 独享一套 KV，表达力最好、KV cache 最大 |
| **GQA**（Grouped-Query Attention） | $1 < G < H$ | 折中：KV cache 降为 $G/H$，质量接近 MHA |
| **MQA**（Multi-Query Attention） | $G = 1$ | 所有 query head 共享一套 KV，KV cache 最小、质量损失最大 |

记 query head 编号为 $h \in \{0, 1, \dots, H-1\}$，则它所用的 key/value head 为

$$
g(h) = \left\lfloor \frac{h}{H/G} \right\rfloor \in \{0, 1, \dots, G-1\}
$$

实现上与 MHA 只有一处不同：key、value 只投影出 $G$ 个 head（而非 $H$ 个），做注意力前再把每个 key/value head **复制** $H/G$ 份、与组内各 query head 对齐；此后的 $\mathrm{softmax}\!\left(QK^\top/\sqrt{d}\right)V$ 与 MHA 完全一致。GQA 以极小的质量代价换来数倍更小的 KV cache，被 Llama 3、Qwen、Gemma、Mistral 等众多现代开源模型广泛采用（DeepSeek-V3 则更进一步，见下方 MLA）。

此外，下面的 `GroupQueryAttn.forward` 还支持 **KV-cache 增量解码**（`past_kv` / `use_cache` 两个参数，默认关闭时行为与原版一致），再往下的代码格会演示它、并验证与整段重算完全一致。

In [ ]:
class GroupQueryAttn(nn.Module):
    def __init__(
        self,
        n_embed,
        head_size,
        num_heads,      # query head 的数量 H
        num_kv_heads,   # key/value head 的数量 G，需能整除 num_heads；=num_heads 退化为 MHA，=1 退化为 MQA
        block_size_max,
    ):
        super().__init__()
        assert num_heads % num_kv_heads == 0, "num_heads 必须能被 num_kv_heads 整除"
        self.n_embed = n_embed
        self.head_size = head_size
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.num_queries_per_kv = num_heads // num_kv_heads # 每个 kv head 被组内多少个 query head 共享（= 需复制的份数 H/G；注意这是“每组的 query 数”，与正文的组数 G=num_kv_heads 相区别）

        # 注意：query 投影出 num_heads 个 head，而 key/value 只投影出 num_kv_heads 个 head（这正是 KV cache 变小的来源）
        self.query = nn.Linear(n_embed, head_size * num_heads,    bias = False)
        self.key   = nn.Linear(n_embed, head_size * num_kv_heads, bias = False)
        self.value = nn.Linear(n_embed, head_size * num_kv_heads, bias = False)
        self.fc = nn.Linear(head_size * num_heads, n_embed) # 将特征维度从 head_size*num_heads 映射回 n_embed
        self.register_buffer("mask", torch.tril(torch.ones(block_size_max, block_size_max)))

    def split_heads(self, h, num_heads):
        batch_size, block_size, total_head_size = h.size()
        assert total_head_size == self.head_size * num_heads
        h = h.reshape(batch_size, block_size, num_heads, self.head_size)
        return h.permute(0, 2, 1, 3) # (batch_size, num_heads, block_size, head_size)

    def forward(self, x, past_kv = None, use_cache = False):
        # x.shape = (batch_size, T_q, n_embed)
        #   · 训练 / 一次性前向：x 是整段序列，past_kv=None、use_cache=False（行为与原版完全一致，只返回 out）
        #   · 推理增量解码：x 只喂“新来的 token”（T_q 通常=1），past_kv 传入上一步的缓存，use_cache=True
        batch_size, T_q, n_embed = x.size()
        assert n_embed == self.n_embed

        q     = self.split_heads(self.query(x), self.num_heads)    # (B, H, T_q, head_size)
        k_new = self.split_heads(self.key(x),   self.num_kv_heads) # (B, G, T_q, head_size)  ← 只有 G 个 kv head
        v_new = self.split_heads(self.value(x), self.num_kv_heads) # (B, G, T_q, head_size)

        # ===== KV cache：把本步新算的 k/v 追加到历史缓存后面 =====
        # 缓存里存的始终是 G 个 kv head（不是 H 个）——这正是 GQA 相对 MHA 省下的显存；历史 k/v 不再重算
        if past_kv is not None:
            past_k, past_v = past_kv
            k = torch.cat([past_k, k_new], dim = 2) # 沿 time 维拼接：历史 + 新 token
            v = torch.cat([past_v, v_new], dim = 2)
        else:
            k, v = k_new, v_new
        present = (k, v) if use_cache else None      # 返回更新后的缓存（仍是 G 个 kv head），供下一步接着用
        T_kv = k.size(2)                              # 当前要 attend 的 key/value 总长度（含历史）
        past_len = T_kv - T_q                         # 历史长度；新 query 的绝对位置从 past_len 开始

        # 到“用的时候”才把 G 份复制成 H 份与 query 对齐（缓存里始终只有 G 份，不占额外显存）
        k_rep = k.repeat_interleave(self.num_queries_per_kv, dim = 1) # (B, H, T_kv, head_size)
        v_rep = v.repeat_interleave(self.num_queries_per_kv, dim = 1)

        w = q @ k_rep.transpose(-2, -1) * (self.head_size ** -0.5)    # (B, H, T_q, T_kv)
        # 因果 mask：第 i 个新 query（绝对位置 past_len+i）只能看 key 位置 ≤ past_len+i
        # 取 mask 的子块 [past_len:past_len+T_q, :T_kv]；整段前向时它就是原来的 [:T,:T]，单 token 解码时则是“能看全部历史+自己”
        w = w.masked_fill(self.mask[past_len:past_len + T_q, :T_kv] == 0, float("-Inf"))
        w = F.softmax(w, dim = -1)
        out = w @ v_rep # (B, H, T_q, head_size)

        out = out.permute(0, 2, 1, 3) # (B, T_q, H, head_size)
        out = out.reshape(batch_size, T_q, self.num_heads * self.head_size)
        out = self.fc(out) # (B, T_q, n_embed)

        # 不开 cache（训练/评估）时返回与原版一致、仅 out；开 cache 时额外返回更新后的缓存 present
        return (out, present) if use_cache else out

group_query_attn = GroupQueryAttn(
    n_embed = n_embed,
    head_size = head_size,
    num_heads = num_heads,   # 8 个 query head
    num_kv_heads = 2,        # 只用 2 个 kv head：8 个 query head 分 2 组、每组 4 个共享 1 个 kv head（num_kv_heads=num_heads 即 MHA，=1 即 MQA）
    block_size_max = block_size,
)

In [ ]:
print("shape of input of group-query attention module: {}".format(x.shape))
print("shape of output of group-query attention module: {}".format(group_query_attn(x).shape))

# query 投影到 num_heads 个 head，而 key/value 只投影到 num_kv_heads 个 head
print("num_heads (query) = {}, num_kv_heads (key/value) = {}".format(group_query_attn.num_heads, group_query_attn.num_kv_heads))
print("KV cache 相比 MHA 缩小为原来的 {}/{} = {:.2f}".format(group_query_attn.num_kv_heads, group_query_attn.num_heads, group_query_attn.num_kv_heads / group_query_attn.num_heads))

In [ ]:
# GQA 的 KV-cache 增量推理演示：逐 token 生成时只缓存 G 个 kv-head 的 K/V、每步追加，
# 并验证结果与“整段重算”完全一致（这才真正把 GQA 省显存的优势用在了推理上）
torch.manual_seed(0)
T = 10
seq = torch.randn(1, T, n_embed)   # 假设这是逐步到来的 T 个 token 的特征

with torch.no_grad():
    # 做法 A（参考）：对整段一次性前向（训练/评估就是这么跑）
    full = group_query_attn(seq)                               # (1, T, n_embed)

    # 做法 B（KV cache 增量解码）：每次只喂“新来的那一个 token”，缓存历史 K/V、逐步追加
    past, steps = None, []
    for t in range(T):
        out_t, past = group_query_attn(seq[:, t:t + 1, :], past_kv=past, use_cache=True)
        steps.append(out_t)                                    # 每步只输出当前这个 token
    incr = torch.cat(steps, dim=1)                             # (1, T, n_embed)

print("KV-cache 增量 == 整段重算:", torch.allclose(full, incr, atol=1e-5),
      "| 最大误差 {:.1e}".format((full - incr).abs().max().item()))

# past 里缓存的 K 是 G 个 kv-head（不是 H 个）——这正是 GQA 省下的显存
past_k, past_v = past
print("缓存 K 形状 (B, G, T, head_size):", tuple(past_k.shape),
      "→ 只存 G={} 个 kv-head，而非 H={} 个".format(group_query_attn.num_kv_heads, group_query_attn.num_heads))
print("每 token KV cache 大小：GQA = 2*G*head_size = {} 个数；MHA 则是 2*H*head_size = {} 个数（约 {:.0f}x 小）".format(
    2 * group_query_attn.num_kv_heads * head_size, 2 * group_query_attn.num_heads * head_size,
    group_query_attn.num_heads / group_query_attn.num_kv_heads))

## 5.5. Multi-Head Latent Attention

**Multi-head Latent Attention（MLA，多头潜注意力）** 是 DeepSeek-V2 / V3 提出的、比 GQA 更激进的省 KV cache 方案。GQA 的思路是“减少 KV head 的数量”（$H \to G $，$H$：head 的数量，$G$：group 的大小）；MLA 换了个角度：**把每个 token 的 K、V 联合压进一个低秩的“潜向量” $c^{KV}$，推理时只缓存这个小向量**。

### 5.5.1. 核心：低秩压缩 + 只缓存潜向量

对每个 token 的输入 $x$：

- **下投影（压缩）**：$c^{KV} = x W_{DKV}$，维度 $d_c$ 很小（$d_c \ll 2 H d_h$）。**推理时缓存的就是这个 $c^{KV}$**。
- **上投影（还原）**：要用时再由 $c^{KV}$ 还原出各 head 的 key、value：$k_i = c^{KV} W_{UK}^{(i)}$、$v_i = c^{KV} W_{UV}^{(i)}$。
- query 侧同样走一次低秩（$x \to c^{Q} \to q$），但那主要是为了省**训练**显存；省 KV cache 的功劳在 $c^{KV}$。

直觉：K、V 的信息高度冗余，用一个低秩瓶颈 $c^{KV}$ 就能“装下”；还原出来的 K、V 仍是满秩多头、表达力损失很小，但要缓存的东西从 $2 H d_h$ 骤降到 $d_c$。

### 5.5.2. 两个让 MLA 真正可用的关键细节

**(1) 解耦 RoPE（decoupled RoPE）**。RoPE 会给 key 按位置乘一个旋转 $R_n$，于是 $k_n = R_n\,(W_{UK}\, c^{KV}_n)$——这个随位置变化的 $R_n$ 卡在 $c^{KV}$ 和 query 之间，使下面的“矩阵吸收”无法进行（低秩压缩与 RoPE 不兼容）。DeepSeek 的解法是把 q、k 各拆成两段：

- **内容段**：走低秩潜向量、**不加 RoPE**（可被吸收）；
- **位置段**：另算一小段带 RoPE 的 query / key，其中 **RoPE key 所有 head 共享一份**（类似 MQA），单独缓存。

注意力打分 = 内容段点积 + 位置段点积。RoPE 只活在那一小段解耦维度上，不碰可压缩的内容路。

**(2) 矩阵吸收（absorption）**——这一步才把“只缓存小向量”真正变成“又省显存又省计算”。核心是一个**只改变矩阵相乘次序**的恒等式：

$$
q_i \cdot \big(W_{UK}^{(i)} c^{KV}\big) = \big(W_{UK}^{(i)\top} q_i\big) \cdot c^{KV}
$$

**单看一次前向，这确实只是换了乘法次序、并不省 FLOPs**。省的关键在**自回归解码 + KV cache 这个循环**里，而循环有一个重要的不对称：**每一步只有 1 个新 query，缓存里却有 $T$ 个 token**。

- **不吸收（等式左边，先还原 K）**：要把 $W_{UK}$ 作用到**每一个缓存 token 的 $c^{KV}$** 上还原成 K——缓存有 $T$ 个就做 $T$ 次上投影；而且因为只缓存了 $c^{KV}$、没缓存 K，**每个解码步都得把这 $T$ 次重做一遍**。
- **吸收（等式右边，把 $W_{UK}$ 挪到 query）**：$\tilde q_i = W_{UK}^{(i)\top} q_i$ 只作用在**那 1 个新 query** 上（每步 1 次、与 $T$ 无关），再直接拿 $\tilde q$ 点乘缓存的 $c^{KV}$。

**省的就是那个 $\times T$**：把上投影从“作用在 $T$ 个缓存 key 上、且每步重做”挪到“作用在 1 个新 query 上、每步一次”。输出侧是同一个道理：

$$
\sum_n a_n \big(W_{UV}^{(i)} c^{KV}_n\big) = W_{UV}^{(i)} \Big(\sum_n a_n\, c^{KV}_n\Big)
$$

左边要先把 $T$ 个 $c^{KV}$ 各还原成 V 再加权；右边则**先在潜向量空间把 $c^{KV}$ 加权聚合成 1 个上下文向量、最后只上投影这一个**——同样把 $W_{UV}$ 从“作用在 $T$ 个缓存 token”挪到了“只作用在 1 个结果上”。

更关键的是**显存与带宽**：既然全程不还原 K/V，缓存里就只放那个 $d_c$ 维的 $c^{KV}$（加共享 RoPE key）。而自回归解码本就是**显存带宽瓶颈**（每步都要把整份 KV cache 从显存读进来），cache 小一个量级、读得就快一个量级——这才是 MLA 在真实推理里提速的主因。

一句话：吸收不是“白改次序”，而是利用解码时“query 少、缓存 key/value 多”的不对称，把上投影挪到**少的那一侧**，省掉原本要在**多的那一侧、每步重复**的计算，顺带把 KV cache 压到最小。

### 5.5.3. KV cache 大小对比

| 方案 | 每个 token 缓存什么 | KV cache 大小 |
|---|---|---|
| **MHA** | 完整 K, V | $2 H d_h$ |
| **GQA** | $G$ 组共享的 K, V | $2 G d_h$ |
| **MLA** | 低秩潜向量 $c^{KV}$（+ 一份共享 RoPE key） | $d_c + d_r$（最小） |

> 下面的 `MultiHeadLatentAttn` 把这三点（低秩压缩 + 解耦 RoPE + 矩阵吸收）都实现了；`forward(x, past_kv, use_cache)` 支持 **KV-cache 增量解码**，缓存里只放 $c^{KV}$ 和那份共享 RoPE key。演示会验证“增量解码 == 整段重算”，并打印出缓存确实只有这两个小向量。
>
> 仍有教学简化：RoPE 用最朴素的 split-half 写法；真实 DeepSeek-V3 还会把 $W_{UK}/W_{UV}$ 离线真正合并进 $W_Q/W_O$、并对缓存做量化等工程优化。此处把“解耦 + 吸收 + 只缓存潜向量”的机理讲清楚即可。

In [ ]:
# MLA 完整实现：低秩压缩(c_KV) + 解耦 RoPE + 矩阵吸收 + KV-cache 增量解码
def apply_rope(t, start_pos = 0):
    # 对 t 的最后一维按“绝对位置 start_pos..start_pos+T-1”做旋转(split-half 约定；维度需偶数)
    # start_pos 让增量解码时新 token 用到正确位置，与整段前向保持一致
    T, d = t.shape[-2], t.shape[-1]
    half = d // 2
    inv = 1.0 / (10000 ** (torch.arange(0, half, device = t.device).float() / half))
    ang = torch.outer(torch.arange(start_pos, start_pos + T, device = t.device).float(), inv)  # (T, half)
    cos, sin = ang.cos(), ang.sin()
    t1, t2 = t[..., :half], t[..., half:]
    return torch.cat([t1 * cos - t2 * sin, t1 * sin + t2 * cos], dim = -1)

class MultiHeadLatentAttn(nn.Module):
    def __init__(
        self,
        n_embed,
        head_size,           # 内容段每个 head 的维度 d_h
        num_heads,           # head 数 H
        block_size_max,
        kv_latent_dim,       # KV 潜向量维度 d_c：推理只缓存它，越小越省(MLA 的核心超参)
        q_latent_dim = None, # query 也走低秩(主要省训练显存)，默认与 kv_latent_dim 相同
        rope_head_dim = 16,  # 解耦 RoPE 段的维度 d_r(很小、需偶数)
    ):
        super().__init__()
        self.n_embed = n_embed
        self.head_size = head_size
        self.num_heads = num_heads
        self.kv_latent_dim = kv_latent_dim
        self.rope_head_dim = rope_head_dim
        q_latent_dim = q_latent_dim or kv_latent_dim

        # query：x -> 低秩 c_Q -> 每个 head 的 [内容段 q_c(head_size) + 位置段 q_r(rope_head_dim)]
        # 说明：下面的 q_down→q_up 是对 query 投影的「低秩分解」。两层之间没有非线性，
        #   数学上等价于单个 Linear(n_embed → num_heads*(head_size+rope_head_dim))，可融合成一层。
        #   真实 MLA 保留它，是为了在 q_latent_dim ≪ n_embed 时省【训练】显存/参数（与 KV cache 无关——q 不缓存）；
        #   本教学版取 q_latent_dim=n_embed，并无压缩效果（分解反而多几百个参数），保留只是为了镜像真实 MLA 的结构。
        self.q_down = nn.Linear(n_embed, q_latent_dim, bias = False)
        self.q_up   = nn.Linear(q_latent_dim, num_heads * (head_size + rope_head_dim), bias = False)
        # 核心：K、V 联合压成一个低秩潜向量 c_KV(kv_latent_dim)——推理时缓存的就是它
        self.kv_down = nn.Linear(n_embed, kv_latent_dim, bias = False)
        self.k_up    = nn.Linear(kv_latent_dim, num_heads * head_size, bias = False)  # W_UK：把 c_KV 还原成各 head 的 K
        self.v_up    = nn.Linear(kv_latent_dim, num_heads * head_size, bias = False)  # W_UV：把 c_KV 还原成各 head 的 V
        # 解耦 RoPE 的 key：直接由 x 投一小段、带 RoPE，【所有 head 共享一份】(类似 MQA)，也缓存它
        self.k_rope  = nn.Linear(n_embed, rope_head_dim, bias = False)
        self.fc      = nn.Linear(num_heads * head_size, n_embed)                      # W_O
        self.register_buffer("mask", torch.tril(torch.ones(block_size_max, block_size_max)))

    def forward(self, x, past_kv = None, use_cache = False):
        # x: (B, T_q, n_embed)
        #   · 训练 / 整段前向：past_kv=None, use_cache=False
        #   · 推理增量解码：x 只喂新 token，past_kv 传入上一步缓存(c_KV 与 共享 RoPE key)，use_cache=True
        B, T_q, _ = x.size()
        H, dh, dc, dr = self.num_heads, self.head_size, self.kv_latent_dim, self.rope_head_dim
        past_len = 0 if past_kv is None else past_kv[0].size(1)   # 已缓存 token 数 = 新 token 的起始绝对位置

        # —— 本步新 token：压出低秩潜向量 c_KV，和一小段共享的解耦 RoPE key(都很小，推理只缓存这两样) ——
        c_kv_new = self.kv_down(x)                        # (B, T_q, dc)
        k_r_new  = apply_rope(self.k_rope(x), past_len)   # (B, T_q, dr)  跨 head 共享
        if past_kv is not None:                           # 追加进缓存(历史不重算)
            c_kv = torch.cat([past_kv[0], c_kv_new], dim = 1)
            k_r  = torch.cat([past_kv[1], k_r_new],  dim = 1)
        else:
            c_kv, k_r = c_kv_new, k_r_new
        present = (c_kv, k_r) if use_cache else None      # 缓存里只放 c_KV 与 共享 RoPE key
        T_kv = c_kv.size(1)

        # —— query：低秩还原后拆成内容段 / 位置段，位置段加 RoPE ——
        q = self.q_up(self.q_down(x)).view(B, T_q, H, dh + dr)
        q_c = q[..., :dh].transpose(1, 2)                       # (B,H,T_q,dh) 内容段
        q_r = apply_rope(q[..., dh:].transpose(1, 2), past_len) # (B,H,T_q,dr) 位置段(带 RoPE)

        # —— 矩阵吸收①：q_c·(W_UK·c_KV) = (W_UK^T·q_c)·c_KV，把 W_UK 并进 query，直接对缓存的 c_KV 打分，不还原 K ——
        W_UK = self.k_up.weight.view(H, dh, dc)
        q_tilde = torch.einsum("bhtd,hdc->bhtc", q_c, W_UK)          # q_tilde = W_UK^T q_c, (B,H,T_q,dc)
        score_content = torch.einsum("bhtc,bsc->bhts", q_tilde, c_kv)
        score_rope    = torch.einsum("bhtd,bsd->bhts", q_r, k_r)     # 位置段(共享 RoPE key)
        w = (score_content + score_rope) * ((dh + dr) ** -0.5)       # (B,H,T_q,T_kv)
        w = w.masked_fill(self.mask[past_len:past_len + T_q, :T_kv] == 0, float("-inf"))
        w = F.softmax(w, dim = -1)

        # —— 矩阵吸收②：Σ a·(W_UV·c_KV) = W_UV·(Σ a·c_KV)，先在潜向量空间聚合，最后只上投影一次，不还原整串 V ——
        ctx  = torch.einsum("bhts,bsc->bhtc", w, c_kv)              # (B,H,T_q,dc)
        W_UV = self.v_up.weight.view(H, dh, dc)
        out  = torch.einsum("bhtc,hdc->bhtd", ctx, W_UV)            # out = W_UV · ctx, (B,H,T_q,dh)
        out  = self.fc(out.transpose(1, 2).reshape(B, T_q, H * dh))
        return (out, present) if use_cache else out

multi_head_latent_attn = MultiHeadLatentAttn(
    n_embed = n_embed,
    head_size = head_size,
    num_heads = num_heads,
    block_size_max = block_size,
    kv_latent_dim = 32,   # 把 K、V 压到 32 维潜向量(远小于 MHA 要缓存的 2*8*16=256)
    rope_head_dim = 16,
)

In [ ]:
# MLA 的 KV-cache 增量推理：逐 token 生成时只缓存“潜向量 c_KV + 共享 RoPE key”(都很小)，每步追加，
# 并验证与“整段重算”逐元素一致——同时看清“全程只缓存小潜向量、K/V 从不显式还原”这一 MLA 优势
torch.manual_seed(0)
T = 10
seq = torch.randn(1, T, n_embed)   # 假设逐步到来的 T 个 token 的特征

with torch.no_grad():
    full = multi_head_latent_attn(seq)                        # 整段一次性前向(参考)
    past, steps = None, []
    for t in range(T):
        out_t, past = multi_head_latent_attn(seq[:, t:t + 1, :], past_kv=past, use_cache=True)
        steps.append(out_t)
    incr = torch.cat(steps, dim=1)

print("MLA KV-cache 增量 == 整段重算:", torch.allclose(full, incr, atol=1e-5),
      "| 最大误差 {:.1e}".format((full - incr).abs().max().item()))

c_kv_cache, k_rope_cache = past
print("缓存内容：潜向量 c_KV {} + 共享 RoPE key {}".format(tuple(c_kv_cache.shape), tuple(k_rope_cache.shape)))
mla_per_tok = multi_head_latent_attn.kv_latent_dim + multi_head_latent_attn.rope_head_dim
mha_per_tok = 2 * num_heads * head_size
print("每 token KV cache：MLA = d_c + d_r = {} 维  vs  MHA = 2*H*head_size = {} 维（约 {:.1f}x 小）".format(
    mla_per_tok, mha_per_tok, mha_per_tok / mla_per_tok))
print("注意：整个过程只缓存/读取这个小潜向量，K、V 从不被显式还原——靠矩阵吸收直接在 c_KV 上做注意力。")

## 5.6. Feed-Forward Layer

在 Attention 模块后添加一个 Feed-Forward Layer。Attention 模块是让不同位置间的特征相互作用，但这样的相互作用是通过对每个位置上的特征和其他位置的特征的加权和得到的，进行了一系列线性变换，捕捉非线性关系的能力有限。故引入 Feed Forward 模块，通过一层线性变换放大维度，过一个激活函数，再用另一层线性变换恢复到原来维度。这种两次线性变换的设计使得 Feed-Forward Layer 可以提取更深入的特征，并通过非线性激活函数增加了模型的表达能力。值得一提的是，FFN 虽然结构简单，却是 Transformer 里参数量与计算量的大头（约占 2/3）——它为每个位置独立地做一次非线性变换，是模型容量与表达力的主要来源之一。

**现代实践**：这里用的激活是 ReLU；现代 LLM 更常用更平滑的 **GELU**（GPT-2 / BERT），或带门控的 **SwiGLU**（Llama 3、Qwen、DeepSeek 等；Google 的 Gemma 用相近的 GeGLU）。SwiGLU 把升维那层拆成两支——一支过 SiLU 当“门”、一支当“值”，逐元素相乘后再降维：$\text{SwiGLU}(x) = \big(\text{SiLU}(xW_1) \odot xW_3\big)W_2$，表达力更强；为在多一支的同时保持参数量相当，其中间维度通常取 $\tfrac{8}{3}d$ 而非 $4d$。


In [ ]:
class FeedForward(nn.Module):
    def __init__(
        self,
        n_embed,
        scaling_factor = 4,
    ):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Linear(n_embed, n_embed * scaling_factor),
            nn.ReLU(),
            nn.Linear(n_embed * scaling_factor, n_embed),
        )
        
    def forward(self, x):
        return self.seq(x)

feed_forward = FeedForward(n_embed = n_embed, scaling_factor = 4)

In [ ]:
print("shape of input: {}".format(x.shape))
print("shape of output of feed forward module: {}".format(feed_forward(multi_head_attn(x)).shape))

In [ ]:
# 现代实践：FFN 的两种常见替代（与上面的 ReLU 版对比）
class FeedForwardGELU(nn.Module):
    # 只把 ReLU 换成更平滑的 GELU（GPT-2 / BERT）
    def __init__(self, n_embed, scaling_factor=4):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Linear(n_embed, n_embed * scaling_factor),
            nn.GELU(),
            nn.Linear(n_embed * scaling_factor, n_embed),
        )
    def forward(self, x):
        return self.seq(x)

class FeedForwardSwiGLU(nn.Module):
    # 门控版（SwiGLU，Llama 3 / Qwen / DeepSeek 等广泛采用）：一支过 SiLU 当“门”、一支当“值”，逐元素相乘后再降维
    def __init__(self, n_embed, hidden=None):
        super().__init__()
        hidden = hidden or int(8 / 3 * n_embed)           # 多一支，中间维取 ~8/3·d，使总参数与 4·d 的普通 FFN 相当
        self.w1 = nn.Linear(n_embed, hidden, bias=False)  # gate 分支
        self.w3 = nn.Linear(n_embed, hidden, bias=False)  # value 分支
        self.w2 = nn.Linear(hidden, n_embed, bias=False)  # 降维回 n_embed
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))   # SiLU(xW1) ⊙ xW3 → W2

# 三者输出形状一致，参数量也相当（SwiGLU 靠 8/3·d 把“多一支”的开销抵掉）
for name, m in [("ReLU   (4d)", FeedForward(n_embed)),
                ("GELU   (4d)", FeedForwardGELU(n_embed)),
                ("SwiGLU (~8/3 d)", FeedForwardSwiGLU(n_embed))]:
    print("{:16s} out={}  params={}".format(name, tuple(m(x).shape), sum(p.numel() for p in m.parameters())))

**更前沿（MoE / 混合专家）**：DeepSeek-V3、Qwen3-MoE、Mixtral 等把这个稠密 FFN 换成 **Mixture-of-Experts (MoE)**——并排放很多个 FFN“专家”，再加一个“路由”网络给每个 token 打分，**每个 token 只激活其中 top-k 个专家**（如 DeepSeek-V3 为 256 选 8、另加若干共享专家）。于是模型的**总参数（容量）可以做得极大，而每个 token 的实际计算量几乎不变**——这是当下把开源模型做大的主流手段。下面直接实现**稀疏分发**：把 token 按选中的专家分组，每个专家只计算分给它的 token，再用 `index_add_` 按路由权重合并输出。每个 token 的专家 FFN 计算量正比于 `top_k`；增加专家总数仍会增加路由打分与分发的开销。

In [ ]:
# 现代实践(MoE)：稀疏分发，每个专家只计算被路由到它的 token
class MoEFeedForward(nn.Module):
    def __init__(self, n_embed, num_experts=4, top_k=2):
        super().__init__()
        self.experts = nn.ModuleList([FeedForward(n_embed) for _ in range(num_experts)])
        self.router = nn.Linear(n_embed, num_experts, bias=False)
        self.top_k = top_k
    def forward(self, x):
        B, T, d = x.shape
        xf = x.reshape(-1, d)                                   # (N, d), N=B*T，把所有 token 摊平
        topw, topi = self.router(xf).topk(self.top_k, dim=-1)   # (N, top_k)
        topw = F.softmax(topw, dim=-1)
        # 摊平成一张“分发表”：N*top_k 条 (token 下标, 选中的专家下标, 权重)
        tok = torch.arange(xf.size(0), device=x.device).repeat_interleave(self.top_k)  # (N*top_k,)
        exp = topi.reshape(-1)                                  # (N*top_k,)
        wgt = topw.reshape(-1, 1)                               # (N*top_k, 1)
        out = torch.zeros_like(xf)
        for e, expert in enumerate(self.experts):               # 每个专家只处理“分给它”的那批 token
            sel = exp == e
            if sel.any():
                t = tok[sel]                                    # 这些 token 的下标
                out.index_add_(0, t, expert(xf[t]) * wgt[sel])  # 算完按权重散射加回去
        return out.view(B, T, d)

moe = MoEFeedForward(n_embed, num_experts=4, top_k=2)
print("MoE 输出形状:", tuple(moe(x).shape))
dense = sum(p.numel() for p in FeedForward(n_embed).parameters())
moe_total = sum(p.numel() for p in moe.parameters())
print("稠密 FFN 参数={} | 4 专家 MoE 总参数={}（约 {:.0f}x），但每个 token 只过 top_k={} 个专家".format(
    dense, moe_total, moe_total / dense, moe.top_k))

**工业级实现怎么做**：上面的稀疏实现还留着一个 `for e in experts` 的 Python 循环、每个专家是一次小 matmul。生产框架会更进一步：

- **分组 GEMM(grouped / segment GEMM)**：把 token 按专家排序后，用一个“分段矩阵乘”一次算完所有专家，消灭 Python 循环（代表：`grouped_gemm`、ScatterMoE）。
- **容量 + 批量 bmm（GShard 风格）**：给每个专家设固定容量，把 token padding 成 `(num_experts, capacity, d)` 用一把 `bmm` 并行算；超容量的 token 被丢弃（近似，换来规整张量形状）。
- **块稀疏 / 融合 kernel**：`Megablocks` 用 block-sparse GEMM 做到“不丢 token 又高效”；vLLM / SGLang 里的 **Triton `fused_moe`** 把 路由 → grouped GEMM → 加权合并 融进极少数 GPU kernel——这才是通常所说的 **“fused MoE”**。
- **专家并行(EP)**：专家很多时把不同专家放到不同 GPU，用 all-to-all 把 token 发过去再收回来（DeepSeek-V3 等大规模 MoE 的标配）。

上述稀疏分发省掉的是**算法层**的冗余（不再算没选中的专家）；工业实现再叠加 **kernel 融合 + 并行**，才能把 MoE“大容量、小计算”的优势真正跑成速度。

## 5.7. Add & LayerNorm

深层神经网络在层数过多时会因为梯度消失或者梯度爆炸产生退化现象，Add（残差连接）使得信息可以跨越一层或者多层，从而缓解梯度消失和梯度爆炸问题。
在介绍 LayerNorm 之前，首先回答，为什么需要归一化（Normalization）？因为有用，实验发现归一化可以平滑 gradients、加快训练、提高模型准确率，但并没有一个确凿的理论解释。比如一种解释是归一化抑制了 Internal Covariate Shift 问题，即深层网络每层都需要适应上层的分布变化，这使得深层的学习变得困难。而归一化让每层的输入分布都保持在一定范围，这样降低模型学习难度。CNN 时代使用的是 BatchNorm。但 LLM 时代改用 LayerNorm：BatchNorm 依赖整个 batch 的统计量，batch 小、或自回归逐 token 推理时都很不稳定（训练/推理行为还不一致）；而 LayerNorm 对每个 token 单独沿特征(dim)维归一化，不依赖 batch，训练和推理行为一致。（序列变长/需要 padding 只是让 batch 维统计更麻烦的因素之一。）然不管哪种 norm，大家都是实验发现有用，然而并不能解释为什么有用。

另外提一句顺序问题：原始 Transformer 是 **Post-LN**（$\text{LayerNorm}(x + \text{Sublayer}(x))$，先相加再归一化）；而下文构建模型时的 Block 用的是现代 LLM 更常用的 **Pre-LN**（$x + \text{Sublayer}(\text{LayerNorm}(x))$，先归一化、再进子层、最后残差相加），它训练更稳、通常无需 learning-rate warmup。此外，所有 Block 堆叠之后还会再接一个 final LayerNorm，把送入输出层的特征再归一化一遍。

**现代实践**：现代 LLM（Llama 3、Qwen、DeepSeek、Gemma 等）多用 **RMSNorm** 替代 LayerNorm——它省去“减均值”和偏置，只按均方根缩放：$\text{RMSNorm}(x) = \dfrac{x}{\sqrt{\frac{1}{d}\sum_i x_i^2 + \epsilon}} \odot g$（$g$ 为可学习的逐维增益）。更省算力，实践中效果与 LayerNorm 相当甚至更稳。

小结一下：如今 **RoPE + GQA + RMSNorm（pre-norm）+ SwiGLU** 已是开源大模型（Llama 3、Qwen、Gemma、Mistral 等）相当统一的“标准底座”；而更前沿的开源模型（DeepSeek-V3、Qwen3-MoE 等）在此之上继续用 **MoE**（扩容量，见上文）与 **MLA**（压 KV cache，见上文）等手段做文章。

In [ ]:
class CheckLayerNorm(nn.Module):
    def __init__(
        self,
        n_embed
    ):
        super().__init__()
        self.ln = nn.LayerNorm(n_embed)
    
    def forward(
        self, x
    ):
        return self.ln(x)

check_layer_norm = CheckLayerNorm(n_embed = n_embed)

In [ ]:
# 观察LayerNorm对特征维度的归一化效果
# x: (batch_size, block_size, n_embed)
ith_sample = 0
ith_token = 0
print("for {}th sample and {}th token in x, the var of feature dim is {} and the mean of feature dim is {}".format(ith_sample, ith_token, x[ith_sample, ith_token].var(), x[ith_sample, ith_token].mean()))
x_after_ln = check_layer_norm(x)
print("after layer normalization, the var of feature dim is {} and the mean of feature dim is {}".format(x_after_ln[ith_sample, ith_token].var(), x_after_ln[ith_sample, ith_token].mean()))

In [ ]:
# 现代实践：RMSNorm（Llama 3 / Qwen / DeepSeek / Gemma 等）vs LayerNorm —— 只按均方根缩放，不减均值、无 bias
class RMSNorm(nn.Module):
    def __init__(self, n_embed, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.g = nn.Parameter(torch.ones(n_embed))    # 可学习的逐维增益（没有 bias，也不减均值）
    def forward(self, x):
        ms = x.pow(2).mean(dim=-1, keepdim=True)       # 沿特征维求“均方”(mean of squares)
        return x * torch.rsqrt(ms + self.eps) * self.g

rms_norm = RMSNorm(n_embed)
ln = nn.LayerNorm(n_embed)
x_ln, x_rms = ln(x), rms_norm(x)
i = (0, 0)   # 看第 0 个样本、第 0 个 token 的特征维
print("输入        : 均值={:.3f} 方差={:.3f}".format(x[i].mean().item(), x[i].var(unbiased=False).item()))
print("LayerNorm 后: 均值={:.3f} 方差={:.3f}  参数={} (weight+bias，既中心化又缩放)".format(
    x_ln[i].mean().item(), x_ln[i].var(unbiased=False).item(), sum(p.numel() for p in ln.parameters())))
print("RMSNorm  后 : 均值={:.3f} RMS={:.3f}  参数={} (仅 gain，不减均值，故均值不必为 0)".format(
    x_rms[i].mean().item(), x_rms[i].pow(2).mean().sqrt().item(), sum(p.numel() for p in rms_norm.parameters())))

# 6. 构建模型

真正训练一个 LLM，除了模型本身，还有一批“工程上几乎必备”的技巧：

- **Dropout（正则化）**：标准 Transformer 会在注意力权重、FFN、残差相加前、embedding 之后加 dropout 防过拟合。玩具规模可默认 `p=0`，需要时在下文定义的 `Attn` / `FeedForward` / `Block` 里插 `nn.Dropout(p)` 即可。
- **权重绑定（weight tying）**：让输入 token embedding 与输出投影共享同一张表——即令 `self.linear_final.weight = self.token_embed_tab.weight`。省参数（BPE 大词表时很可观）、常还能提效；直觉是“查表编码”与“解码打分”互为逆操作。
- **权重初始化**：本教程用 PyTorch 默认初始化；GPT-2 则用正态 $N(0,\,0.02)$，并把残差分支的投影再按 $1/\sqrt{2\,n_\text{layers}}$ 缩放，对深层训练稳定性有实质帮助。
- **学习率调度**：下文用固定 `lr=1e-4`；现代 LLM 几乎都用 **warmup + 余弦退火**（先线性升温防早期发散，再余弦衰减助收敛）。
- **梯度裁剪**：下文的训练循环里已内置一行 `clip_grad_norm_`，这是防梯度爆炸最直接的手段，与上文的残差 / 归一化互补。
- **特殊 token（BOS / EOS / PAD）**：真实 LLM 用特殊 token 标记序列**开始**（BOS）、**结束**（EOS，呼应前文——生成出 EOS 即停）、以及批内对齐的**填充**（PAD，算 loss 时用 `ignore_index` 忽略）。本教程是定长字符级、从固定起点生成到固定长度，故省去；换成通用 / 

用上以上技巧，我们来构建模型：

In [ ]:
class Attn(nn.Module):
    def __init__(
        self,
        n_embed, head_size, num_heads, block_size_max, dropout = 0.0
    ):
        super().__init__()
        self.head_size = head_size
        self.num_heads = num_heads
        self.n_embed = n_embed

        self.key = nn.Linear(n_embed, head_size * num_heads, bias = False)
        self.query = nn.Linear(n_embed, head_size * num_heads, bias = False)
        self.value = nn.Linear(n_embed, head_size * num_heads, bias = False)
        self.fc = nn.Linear(head_size * num_heads, n_embed)
        self.attn_dropout = nn.Dropout(dropout)   # 作用在注意力权重(softmax 之后)
        self.resid_dropout = nn.Dropout(dropout)  # 作用在输出投影之后(残差分支)
        self.register_buffer("mask", torch.tril(torch.ones(block_size_max, block_size_max)))

    def split_heads(self, h):
        batch_size, seq_length, total_embed_size = h.size()
        assert total_embed_size == self.head_size * self.num_heads
        h = h.view(batch_size, seq_length, self.num_heads, self.head_size)
        h = h.permute(0, 2, 1, 3)
        return h

    def forward(self, x, k = None, v = None, mask = None, past_kv = None, use_cache = False):
        # 仅投影本次输入；历史 K/V 已是 (B, H, T_past, head_size)。
        batch_size, seq_length, n_embed = x.size()
        assert n_embed == self.n_embed
        if past_kv is not None and not use_cache:
            raise ValueError("传入 past_kv 时必须启用 use_cache")
        if (k is None) != (v is None):
            raise ValueError("k 和 v 必须同时提供")
        if k is None:
            k, v = self.key(x), self.value(x)
        elif k.shape != v.shape or k.ndim != 3 or k.shape[0] != batch_size or k.shape[2] != self.head_size * self.num_heads:
            raise ValueError("k/v 应为形状相同的 (B, T, H * head_size) 投影")
        if use_cache and k.shape[1] != seq_length:
            raise ValueError("KV cache 用于自注意力，本次 Q/K/V 的 token 数应一致")
        k, v = self.split_heads(k), self.split_heads(v)
        q = self.split_heads(self.query(x))

        past_len = 0
        if past_kv is not None:
            if not isinstance(past_kv, (tuple, list)) or len(past_kv) != 2:
                raise ValueError("每层 past_kv 应为 (past_k, past_v)")
            past_k, past_v = past_kv
            if not isinstance(past_k, torch.Tensor) or not isinstance(past_v, torch.Tensor):
                raise ValueError("past_k 和 past_v 必须是 Tensor")
            if past_k.ndim != 4 or past_k.shape != past_v.shape or past_k.shape[:2] != (batch_size, self.num_heads) or past_k.shape[-1] != self.head_size:
                raise ValueError("KV cache 形状应为 (B, H, T_past, head_size)")
            if past_k.device != k.device or past_v.device != v.device or past_k.dtype != k.dtype or past_v.dtype != v.dtype:
                raise ValueError("KV cache 与当前 K/V 的 device、dtype 必须一致")
            past_len = past_k.shape[2]
            k = torch.cat((past_k, k), dim=2)
            v = torch.cat((past_v, v), dim=2)

        key_len = k.shape[2]
        if past_len + seq_length > self.mask.shape[0] or key_len > self.mask.shape[1]:
            raise ValueError("注意力长度超过 block_size_max")
        w = q @ k.transpose(-2, -1) * (self.head_size ** -0.5)
        if mask is not None:
            if mask.shape != (seq_length, key_len):
                raise ValueError("mask 形状应为 (T_new, T_past + T_new)")
        else:
            # 查询对应位置 past_len ... past_len+T_new-1；支持一次追加多个 token。
            mask = self.mask[past_len:past_len + seq_length, :key_len]
        w = w.masked_fill(mask == 0, float("-inf"))
        w = self.attn_dropout(F.softmax(w, dim=-1))
        out = w @ v
        out = out.permute(0, 2, 1, 3).reshape(batch_size, seq_length, self.head_size * self.num_heads)
        out = self.resid_dropout(self.fc(out))
        return (out, (k, v)) if use_cache else out


class FeedForward(nn.Module):
    def __init__(self, n_embed, scaling_factor = 4, dropout = 0.0):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Linear(n_embed, n_embed * scaling_factor),
            nn.ReLU(),
            nn.Linear(n_embed * scaling_factor, n_embed),
            nn.Dropout(dropout),   # 默认 p=0 时为恒等
        )

    def forward(self, x):
        return self.seq(x)


class Block(nn.Module):
    def __init__(
        self,
        n_embed, head_size, num_heads, block_size_max, dropout = 0.0
    ):
        super().__init__()
        self.multiheadattn = Attn(n_embed, head_size, num_heads, block_size_max, dropout = dropout)
        self.ffwd = FeedForward(n_embed, dropout = dropout)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x, past_kv = None, use_cache = False):
        # Pre-Norm：缓存的是该层 ln1(x) 投影后的 K/V。
        attn_out = self.multiheadattn(self.ln1(x), past_kv = past_kv, use_cache = use_cache)
        if use_cache:
            attn_out, present_kv = attn_out
        x = x + attn_out
        x = x + self.ffwd(self.ln2(x))
        return (x, present_kv) if use_cache else x


class Model(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embed,
        n_layers,
        head_size,
        num_heads,
        block_size,
        block_size_max,
        dropout = 0.0
    ):
        super().__init__()
        self.token_embed_tab = nn.Embedding(vocab_size, n_embed)
        self.pos_embed_tab = nn.Embedding(block_size_max, n_embed)
        self.drop = nn.Dropout(dropout)   # embedding 之后的 dropout
        self.blocks = nn.Sequential(*[Block(n_embed = n_embed, head_size = head_size, num_heads = num_heads, block_size_max = block_size_max, dropout = dropout) for _ in range(n_layers)])
        self.layernorm_final = nn.LayerNorm(n_embed)
        self.linear_final = nn.Linear(n_embed, vocab_size)
        self.block_size = block_size

        # 权重绑定(weight tying)：输入 token embedding 与输出投影共享同一张表
        # （二者形状都是 (vocab_size, n_embed)，直觉上“查表编码”与“解码打分”互为逆操作）
        self.linear_final.weight = self.token_embed_tab.weight

        # 权重初始化：所有 Linear / Embedding 用 N(0, 0.02)
        self.apply(self._init_weights)
        # GPT-2 的小技巧：残差分支的输出投影再按 1/sqrt(2*n_layers) 缩放
        for pn, p in self.named_parameters():
            if pn.endswith("fc.weight") or pn.endswith("seq.2.weight"):
                torch.nn.init.normal_(p, mean = 0.0, std = 0.02 * (2 * n_layers) ** -0.5)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)

    def _cache_length(self, past_kv, batch_size):
        if past_kv is None:
            return 0
        if not isinstance(past_kv, (tuple, list)) or len(past_kv) != len(self.blocks):
            raise ValueError("past_kv 必须为每个 Transformer 层各提供一对 K/V")
        lengths = []
        for block, layer_kv in zip(self.blocks, past_kv):
            if not isinstance(layer_kv, (tuple, list)) or len(layer_kv) != 2:
                raise ValueError("每层 cache 应为 (K, V)")
            k, v = layer_kv
            attn = block.multiheadattn
            if not isinstance(k, torch.Tensor) or not isinstance(v, torch.Tensor):
                raise ValueError("缓存的 K/V 必须是 Tensor")
            if k.ndim != 4 or k.shape != v.shape or k.shape[:2] != (batch_size, attn.num_heads) or k.shape[-1] != attn.head_size:
                raise ValueError("每层 KV cache 形状应为 (B, H, T_past, head_size)")
            lengths.append(k.shape[2])
        if lengths and any(length != lengths[0] for length in lengths):
            raise ValueError("所有层的 KV cache 长度必须一致")
        return lengths[0] if lengths else 0

    def forward(self, x, target = None, past_kv = None, use_cache = False):
        if x.ndim != 2 or x.shape[0] == 0 or x.shape[1] == 0:
            raise ValueError("x 应为非空的 (B, T) token 序列")
        if target is not None and use_cache:
            raise ValueError("训练时不要启用 KV cache；使用 model(x, target)")
        if past_kv is not None and not use_cache:
            raise ValueError("传入 past_kv 时必须启用 use_cache")
        B, T = x.shape
        past_len = self._cache_length(past_kv, B)
        if past_len + T > self.pos_embed_tab.num_embeddings:
            raise ValueError("总长度超过学习式位置编码的 block_size_max")
        tok_embed = self.token_embed_tab(x)
        # 增量输入的位置从历史缓存长度继续编号。
        pos_embed = self.pos_embed_tab(torch.arange(past_len, past_len + T, device = x.device))
        x = self.drop(tok_embed + pos_embed)
        if use_cache:
            presents = []
            for layer_idx, block in enumerate(self.blocks):
                layer_past = None if past_kv is None else past_kv[layer_idx]
                x, layer_present = block(x, past_kv = layer_past, use_cache = True)
                presents.append(layer_present)
        else:
            x = self.blocks(x)
        logits = self.linear_final(self.layernorm_final(x))

        if target is None:
            loss = None
        else:
            assert target.shape == logits.shape[:-1]
            loss = F.cross_entropy(logits.view(B*T, -1), target.view(B*T))
        return (logits, loss, tuple(presents)) if use_cache else (logits, loss)

    def decode_step(self, context, past_kv = None, use_cache = True, block_size = None):
        """默认 KV cache；返回 (B, V) logits 与 cache。

        past_kv 必须来自同一请求的上一步，context 只在末尾新增了一个 token；
        新请求或修改已有上下文时传 None。首步 prefill，随后仅输入新增 token。
        """
        if context.ndim != 2 or context.shape[0] == 0 or context.shape[1] == 0:
            raise ValueError("context 应为非空的 (B, T) token 序列")
        window = self.block_size if block_size is None else block_size
        if not isinstance(window, int) or not 0 < window <= self.pos_embed_tab.num_embeddings:
            raise ValueError("block_size 必须在 [1, block_size_max] 内")
        if not use_cache:
            logits, _ = self(context[:, -window:])
            return logits[:, -1, :], None
        past_len = self._cache_length(past_kv, context.shape[0])
        if past_len > window:
            raise ValueError("缓存长度超过当前推理窗口，请从 past_kv=None 重新开始")
        if past_kv is not None:
            if past_len < window and context.shape[1] != past_len + 1:
                raise ValueError("context 必须为缓存对应的上下文末尾新增一个 token；新请求请传 past_kv=None")
            if past_len == window and context.shape[1] <= window:
                raise ValueError("缓存窗口已满，context 必须包含新增 token 及完整滑动窗口")
        if past_kv is not None and past_len + 1 <= window:
            logits, _, presents = self(context[:, -1:], past_kv = past_kv, use_cache = True)
        else:
            # 首步或窗口满时重建。位置从 0 重置，深层 K/V 也依赖被丢弃的上下文，
            # 因而不能只裁剪旧 cache；重建才能与原来的滑动窗口推理严格一致。
            logits, _, presents = self(context[:, -window:], use_cache = True)
        return logits[:, -1, :], presents

    def generate(self, x, max_new_tokens = 100, use_cache = True, eos_token_id = None):
        """默认使用 KV cache；use_cache=False 可作为全窗口重算的对照。"""
        if x.ndim != 2 or x.shape[0] == 0 or x.shape[1] == 0:
            raise ValueError("prompt 应为非空的 (B, T) token 序列")
        if not isinstance(max_new_tokens, int) or max_new_tokens < 0:
            raise ValueError("max_new_tokens 必须为非负整数")
        if eos_token_id is not None and (not isinstance(eos_token_id, int) or not 0 <= eos_token_id < self.token_embed_tab.num_embeddings):
            raise ValueError("eos_token_id 必须为词表内的 token id")
        was_training = self.training
        self.eval()
        try:
            with torch.inference_mode():
                past_kv = None
                finished = torch.zeros(x.shape[0], dtype=torch.bool, device=x.device)
                for _ in range(max_new_tokens):
                    logits, past_kv = self.decode_step(x, past_kv = past_kv, use_cache = use_cache)
                    probs = F.softmax(logits, dim=-1)
                    idx_next = torch.multinomial(probs, num_samples=1)
                    if eos_token_id is not None:
                        # 同一批次保持矩形：已结束样本继续填 EOS，全部结束时退出。
                        idx_next = idx_next.masked_fill(finished[:, None], eos_token_id)
                        finished |= idx_next[:, 0] == eos_token_id
                    x = torch.cat((x, idx_next), dim=1)
                    if eos_token_id is not None and finished.all():
                        break
        finally:
            self.train(was_training)
        return x

除去模型本身、工程技巧、输入输出数据，我们还需要 **损失函数** 来度量模型当前输出和目标结果的区别，我们用 **交叉熵（Cross-Entropy）** 作损失函数（已经在上述代码中实现 `loss = F.cross_entropy(logits.view(B*T, -1), target.view(B*T))`）。

## 6.1. 为什么用交叉熵（Cross-Entropy）作损失函数？

模型每个位置输出的，是“下一个 token 在整个词表上的概率分布” $p_\theta(\cdot \mid \text{context}) = \mathrm{softmax}(\text{logits})$。训练的目标就是：让模型给**真实出现的那个下一个 token** 尽量高的概率。交叉熵正是把这句话写成一个可优化的损失。

**它本质上就是最大似然（MLE）。** 把某个位置真实的下一个 token 记作 $y$，该位置的交叉熵损失是

$$
\ell = -\log p_\theta(y \mid \text{context})
$$

在全部 $B\times T$ 个位置上平均：

$$
\mathcal{L} = -\frac{1}{BT}\sum_{\text{所有位置}} \log p_\theta(y \mid \text{context})
$$

最小化 $\mathcal{L}$ $\Longleftrightarrow$ 最大化模型赋予整份训练数据的对数似然 $\Longleftrightarrow$ **极大似然估计**。这正是前文 $p(t_k \mid t_1,\dots,t_{k-1}, \theta)$ 的训练版：把概率质量尽量压到真实出现的 token 上。

**信息论视角：这里的交叉熵就等于 KL 散度。** 记真实分布 $q$ 为 one-hot（只有 $y$ 处为 1），模型分布为 $p$，则

$$
H(q, p) = -\sum_i q_i \log p_i = -\log p_y
$$

又有恒等式 $H(q,p) = H(q) + D_{\mathrm{KL}}(q \parallel p)$，而 one-hot 分布的熵 $H(q)=0$，于是在“目标是 one-hot”的语言建模里，**交叉熵恰好等于 KL 散度** $D_{\mathrm{KL}}(q \parallel p)$——最小化它，就是把模型分布 $p$ 往数据分布 $q$ 拉近。而 $-\log p_y$ 本身是“真实 token 在模型眼里的**惊讶度 / 信息量**”，训练就是最小化平均惊讶。

**直觉：**
- 答对且自信（$p_y \to 1$）→ loss $\to 0$；而**答错却自信（$p_y \to 0$）→ loss $\to +\infty$**——对“自信地犯错”重罚，逼模型诚实地给出概率。
- 它只盯着“正确 token 拿到多少概率”，不关心其余 token 之间怎么瓜分——正好契合 next-token 任务。

**为什么不用 MSE 或准确率？**
- 准确率不可导，没法反向传播。
- 对分类用 MSE，梯度在 softmax 饱和区会趋近 0、学得很慢；而 **softmax + 交叉熵的梯度特别干净**：$\partial \mathcal{L} / \partial\, \text{logit}_i = p_i - q_i$（预测概率 − one-hot 目标），数值稳定、好优化。

**perplexity $= e^{\mathcal{L}}$** 则是交叉熵的“人话版”：≈ 模型每步平均要在多少个候选里纠结（就是上面 `val()` 里打印的那个）。下面用几行代码把这些逐一验证：

In [ ]:
# 为什么用交叉熵：几个一行就能验证的事实
torch.manual_seed(0)
V = 10                       # 假设词表大小为 10
logits = torch.randn(1, V, requires_grad=True)
target = torch.tensor([3])   # 真实的下一个 token 是第 3 个

# (1) 交叉熵 = -log( 模型赋予“正确 token”的概率 ) —— 即最大似然
ce = F.cross_entropy(logits, target)
p  = F.softmax(logits, dim=-1)
print("F.cross_entropy    =", ce.item())
print("-log p[正确 token] =", (-torch.log(p[0, target])).item(), " (两者相同)")

# (2) softmax + 交叉熵的梯度特别干净：dL/d(logits) = p - onehot(target)
ce.backward()
onehot = F.one_hot(target, V).float()
print("梯度 == p - onehot ?", torch.allclose(logits.grad, p - onehot, atol=1e-6))

# (3) 在这里 交叉熵 == KL(真实 one-hot || 模型 p)（因为 one-hot 的熵为 0）
kl = (onehot * (torch.log(onehot + 1e-12) - torch.log(p))).sum()
print("交叉熵 == KL(真实||模型) ?", torch.allclose(ce, kl, atol=1e-6))

# (4) loss 只取决于“给正确 token 的概率”：越自信越对→0；越自信越错→∞（重罚自信的错）
for pc in [0.99, 0.5, 0.1, 0.01]:
    print("  p(正确)={:.2f} -> loss = -log p = {:.3f}".format(pc, -torch.log(torch.tensor(pc)).item()))

# (5) perplexity = exp(交叉熵)：≈ 平均要在多少个候选里纠结
print("若平均交叉熵 = 2.0 -> perplexity = exp(2.0) =", round(torch.exp(torch.tensor(2.0)).item(), 2))

## 7. 训练

上面的 `Model` 其实同时写好了两种“跑法”，值得停下来对比——这是初学者最容易混淆的点：

- **训练（`forward`，并行、teacher forcing）**：一条长为 $T$ 的序列喂进去，模型在**一次前向**里就对全部 $T$ 个位置同时做 next-token 预测，靠因果 mask 保证第 $i$ 个位置只看得到它及之前的真实 token。每个位置的条件都是**真实前缀**（而非模型自己猜的），这叫 **teacher forcing**；cross-entropy 在全部 $B\times T$ 个位置上平均，于是一条序列能产生 $T$ 条监督信号，训练非常高效。
- **推理（`generate`，串行、自回归）**：没有“真值”可喂，只能把模型自己采样出的 token **回灌**到输入末尾、再预测下一个，一个一个地生成（autoregressive）。本模型默认先并行处理 prompt（prefill），随后用 KV cache 逐 token 增量生成（decode）；普通自回归循环的新 token 仍依次产生。前面的采样结果会影响后续输出；后续还会介绍可减少目标模型调用次数的推测解码。

一句话：**训练时并行地“看着答案”学所有位置，推理时串行地“蒙着答案”一个一个写**。

下面正式开始训练

init model

In [ ]:
model = Model(
    vocab_size,
    n_embed,
    n_layers,
    head_size,
    num_heads,
    block_size,
    block_size_max = block_size
)
model = model.to(device)

init optimizer

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

**示例：learning-rate warmup + 余弦退火**

上面的训练用的是固定 `lr=1e-4`。下面给一个可接入的调度：先用 `warmup_steps` 步把 lr 从接近 0 线性升到峰值（防早期发散），之后余弦衰减到一个最小值。画条曲线看看形状：

In [ ]:
import math

def get_lr(step, max_lr=1e-4, min_lr=1e-5, warmup_steps=100, max_steps=2000):
    if step < warmup_steps:                                      # 1) 线性 warmup：~0 -> max_lr
        return max_lr * (step + 1) / warmup_steps
    if step > max_steps:                                         # 2) 结束后保持最小 lr
        return min_lr
    ratio = (step - warmup_steps) / (max_steps - warmup_steps)   # 3) 余弦退火：max_lr -> min_lr
    return min_lr + 0.5 * (1 + math.cos(math.pi * ratio)) * (max_lr - min_lr)

plt.figure()
plt.plot([get_lr(s) for s in range(2200)])
plt.xlabel("step"); plt.ylabel("lr"); plt.title("warmup + cosine schedule"); plt.show()

# 接入训练循环的方式：每步开始前，用算好的 lr 覆盖 optimizer 的学习率
#   lr = get_lr(global_step)
#   for g in optimizer.param_groups:
#       g["lr"] = lr

validation

In [ ]:
@torch.no_grad()
def val():
    losses = []
    model.eval()
    for batch in tqdm(test_datasetloader):
        logits, loss = model(
            batch["input"].to(device),
            batch["target"].to(device),
        )
        losses.append(loss.item())
    model.train()
    avg_loss = sum(losses) / len(losses)
    print(f"val perplexity: {torch.exp(torch.tensor(avg_loss)).item():.2f}")  # perplexity = exp(交叉熵)，LM 的标准指标：≈“模型每步平均要在多少个候选里纠结”
    return avg_loss

def mov_avg(value, last_value, alpha = 0.99):
    return last_value * alpha + value * (1 - alpha)

In [ ]:
mov_avg_loss = None
for ith_epoch in range(num_epoch):
    for i, batch in enumerate(train_datasetloader):
        logits, loss = model(
            batch["input"].to(device),
            batch["target"].to(device),
        )
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # 梯度裁剪：防梯度爆炸的标准安全网
        optimizer.step()
        if mov_avg_loss is None:
            mov_avg_loss = mov_avg(loss.item(), loss.item())
        else:
            mov_avg_loss = mov_avg(loss.item(), mov_avg_loss)
        if (i+1) % print_interval == 0:
            print(f"epoch {ith_epoch+1} | step {i+1} | cur train loss: {loss.item()} | mov avg train loss: {mov_avg_loss}")

    print("start eval")
    val_loss = val()
    print(f"val loss: {val_loss}")

In [ ]:
# 默认 use_cache=True：窗口内增量解码；超过窗口后重建（详见下文）
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print("[model output]:\n\n{}".format("".join([decode(i) for i in model.generate(context, max_new_tokens = 1000).detach().cpu().numpy()])))

In [ ]:
# 不使用 KV cache 的对照写法；默认生成请调用 model.generate
# context = torch.zeros((1, 1), dtype=torch.long, device=device)
# max_new_tokens = 5000
# for _ in range(max_new_tokens):
#     with torch.no_grad():
#         logits, _ = model(context[:, -block_size:])
#     probs = F.softmax(logits[:, -1, :], dim=-1)
#     idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
#     context = torch.cat((context, idx_next), dim=1) # (B, T+1)

In [ ]:
# "".join([decode(i) for i in context.detach().cpu().numpy()])

# 8. 推理技巧

## 8.1. 解码策略

模型每一步输出的，是下一个 token 在整个词表上的一个概率分布。那么——该如何从这个分布里挑出 `next token`？难道只会挑概率最大的那个吗？下面介绍几种常用的解码策略。

- **Greedy search（贪心）**：每步都取概率最大的 token（`argmax`）。确定性强、容易复现，但也最容易陷入重复乏味的循环（如 “the the the …”），因为它从不冒险。

- **Temperature（温度）**：采样前先把 logits 除以一个温度 $T$ 再做 softmax。$T$ 越大，分布越“平”、采样越多样、越可能出意外；$T$ 越小分布越“尖”、越接近 greedy（$T\to 0$ 即 greedy，$T=1$ 为原始分布）。

- **Top-k sampling**：只在概率最高的 $k$ 个候选里按概率随机采样，其余置零。既保留随机性，又避免采到长尾里的离谱 token。

- **Nucleus / Top-p sampling**：不固定候选个数，而是取“累积概率刚好达到 $p$ 的最小 token 集合”——这个集合大小会随分布自适应（分布尖时候选少、平时候选多），再在其中按概率采样。

- **Beam search（束搜索）**：同时维护 $b$ 条候选序列（“束”），每步按累积 log-概率扩展、只保留最优的 $b$ 条。常用于机器翻译 / 摘要这类有“较优解”的任务；对开放式创作型生成反而容易重复乏味，现代 LLM 很少用，这里不展开。

想更系统地了解，可参考[How to generate text: using different decoding methods for language generation with Transformers](https://huggingface.co/blog/how-to-generate)

## 8.2. 默认使用 KV cache 的生成循环

**本节的 `GreedySearch`、`TopkSampling`、`NucleusSampling`、EOS 示例，以及之前的 `Model.generate`，均默认 `use_cache=True`。** 采样策略和缓存相互独立：前者决定怎样选 token，后者避免重复计算。显式传 `use_cache=False` 可对照整窗口重算。

第一步 **prefill** 处理 prompt、建立**每个 Transformer 层**的 K/V 缓存，取最后一个位置的 logits 选出第一个新 token；之后 **decode** 只把上一步生成的一个新 token 喂入模型。`Model.forward(..., past_kv=..., use_cache=True)` 返回 `(logits, loss, past_kv)`，每层缓存形状为 `(B, num_heads, T_cached, head_size)`。增量位置从缓存长度继续编号，因果 mask 按这个偏移截取；训练仍用 `model(x, target)`，默认不缓存。

下面的生成循环还会设置 `model.eval()` 并进入 `torch.inference_mode()`：前者关闭 dropout，后者关闭梯度记录，结束后恢复调用前的模型模式。每个请求独立维护缓存；缓存必须对应同一条未修改的前缀，不能拿别的 prompt 的 KV 接着生成。

**本教程的上下文限制**：模型使用可学习绝对位置编码，默认 `block_size=16`。首步只处理最近 `block_size` 个 prompt token；窗口内增量复用 KV。窗口满后继续生成会丢弃最早 token、把剩余位置重新从 0 编号；既有 K/V 的位置和深层表示都已不匹配，因而 `decode_step` 会**重建整个窗口的缓存**，保持与原先滑动窗口重算相同的 logits。超过窗口后不再享有逐 token 增量加速，仅裁掉旧 KV 也不能保持相同结果。真实 LLM 通常在更大的支持窗口内持续复用缓存；滑动窗口模型和长期缓存需要相应的位置、注意力设计。

In [ ]:
# 三种策略共享 KV-cache 生成循环；采样函数只决定“如何选下一个 token”
class _TextDecoder:
    def __init__(self, model, encode, decode, block_size, device):
        self.model = model
        self.encode = encode
        self.decode = decode
        self.block_size = block_size
        self.device = device

    def _generate(self, input_text_list, sample, max_new_tokens, use_cache, eos_token_id, desc):
        if isinstance(input_text_list, str):
            input_text_list = [input_text_list]
        if not isinstance(input_text_list, list) or not all(isinstance(t, str) for t in input_text_list):
            raise ValueError("输入须为字符串或字符串列表")
        if max_new_tokens < 0:
            raise ValueError("max_new_tokens 必须非负")
        output_text_list = []
        was_training = self.model.training
        self.model.eval()  # 关闭 dropout；inference_mode 另负责关闭梯度记录
        try:
            with torch.inference_mode():
                # 按单条序列依次处理；这是教学循环，多请求并行调度见本节末尾
                for i, text in enumerate(input_text_list):
                    context = torch.tensor(self.encode(text), dtype=torch.long, device=self.device).unsqueeze(0)
                    if context.shape[1] == 0:
                        raise ValueError("prompt 至少需要一个 token；空输入可显式加入 BOS")
                    past_kv = None  # 每个 prompt 有独立缓存，不在不同请求间串用
                    for _ in tqdm(range(max_new_tokens), desc=f"[{i+1}|{len(input_text_list)}] {desc}"):
                        next_token_logits, past_kv = self.model.decode_step(
                            context, past_kv=past_kv, use_cache=use_cache, block_size=self.block_size)
                        idx_next = sample(next_token_logits)  # (1, 1)
                        context = torch.cat((context, idx_next), dim=1)
                        if eos_token_id is not None and idx_next.item() == eos_token_id:
                            break
                    output_text_list.append(self.decode(context[0].tolist()))
        finally:
            self.model.train(was_training)
        return output_text_list


# Greedy：直接取 logits 最大的 token；默认 use_cache=True
class GreedySearch(_TextDecoder):
    def __call__(self, input_text_list, max_new_tokens=100, use_cache=True, eos_token_id=None):
        return self._generate(
            input_text_list, lambda logits: logits.argmax(dim=-1, keepdim=True),
            max_new_tokens, use_cache, eos_token_id, "decoding by greedy search")


greedy_search = GreedySearch(model, encode, decode, block_size, device)

In [ ]:
input_text = "JACKAL:"
print("[model input]:\n\n{}\n\n=====\n\n[model output]:\n\n{}\n\n=====\n\n".format(input_text, greedy_search(input_text, max_new_tokens = 100)[0]))

In [ ]:
input_text_list = ["JACKAL:", "Antonio:"]
for i, out in enumerate(greedy_search(input_text_list, max_new_tokens = 100)):
    print("#{}#\n\n[model input]:\n\n{}\n\n=====\n\n[model output]:\n\n{}\n\n=====\n\n".format(i+1, input_text_list[i], out))

In [ ]:
# Top-k sampling：默认 use_cache=True，温度须 > 0（贪心请用 GreedySearch）
class TopkSampling(_TextDecoder):
    def __call__(self, input_text_list, top_k=5, temperature=1, max_new_tokens=100,
                 use_cache=True, eos_token_id=None):
        if top_k < 1 or temperature <= 0:
            raise ValueError("top_k 必须 >= 1，temperature 必须 > 0")

        def sample(logits):
            if top_k > logits.shape[-1]:
                raise ValueError("top_k 不能超过词表大小")
            # 第 k 大以外的 logits 置为 -inf，然后在保留的候选里按温度采样
            kth_value = logits.topk(top_k, dim=-1).values[:, -1:]
            logits = logits.masked_fill(logits < kth_value, float('-inf'))
            probs = F.softmax(logits / temperature, dim=-1)
            return torch.multinomial(probs, 1)

        return self._generate(input_text_list, sample, max_new_tokens, use_cache,
                              eos_token_id, "decoding by top-k sampling")


topk_sampling = TopkSampling(model, encode, decode, block_size, device)

In [ ]:
input_text = "JACKAL:"
print("[model input]:\n\n{}\n\n=====\n\n[model output]:\n\n{}\n\n=====\n\n".format(input_text, topk_sampling(input_text, top_k = 10, temperature = 1, max_new_tokens = 100)[0]))

In [ ]:
input_text_list = ["JACKAL:", "Antonio:"]
for i, out in enumerate(topk_sampling(input_text_list, top_k = 10, temperature = 1, max_new_tokens = 100)):
    print("#{}#\n\n[model input]:\n\n{}\n\n=====\n\n[model output]:\n\n{}\n\n=====\n\n".format(i+1, input_text_list[i], out))

In [ ]:
# Nucleus / Top-p sampling：默认 use_cache=True
class NucleusSampling(_TextDecoder):
    def __call__(self, input_text_list, p=0.9, temperature=1, max_new_tokens=100,
                 use_cache=True, eos_token_id=None):
        if not 0 < p <= 1 or temperature <= 0:
            raise ValueError("p 必须在 (0, 1] 内，temperature 必须 > 0")

        def sample(logits):
            sorted_logits, sorted_idxs = logits.sort(dim=-1, descending=True)
            sorted_probs = F.softmax(sorted_logits / temperature, dim=-1)
            cumsum = sorted_probs.cumsum(dim=-1)
            sorted_remove = cumsum > p
            # 右移一位，保留“刚好越过 p 的边界 token”，且至少保留第一个候选
            sorted_remove[:, 1:] = sorted_remove[:, :-1].clone()
            sorted_remove[:, 0] = False
            remove = torch.zeros_like(sorted_remove).scatter(1, sorted_idxs, sorted_remove)
            logits = logits.masked_fill(remove, float('-inf'))
            return torch.multinomial(F.softmax(logits / temperature, dim=-1), 1)

        return self._generate(input_text_list, sample, max_new_tokens, use_cache,
                              eos_token_id, "decoding by nucleus sampling")


nucleus_sampling = NucleusSampling(model, encode, decode, block_size, device)

**示例：遇到 EOS 就停（默认 KV cache）**

本教程默认是字符级、没有真正的结束符；`Model.generate` 和三种采样器都支持可选的 `eos_token_id`，不传时跑满 `max_new_tokens`。真实 LLM 应使用 tokenizer / 模型配置指定的 EOS 或回合结束 token。下面复用 `Model.generate` 的缓存循环，字符级仅拿**换行符**的 id 演示停止逻辑；它不是真实 EOS。采样器也可以用同样的参数提前停止。

In [ ]:
# 复用默认 KV-cache 生成循环，避免 EOS 示例另写一套全窗口重算
# idx 为 (1, T)；Model.generate 也支持等长 batch，并为已结束样本填充 EOS
def generate_until_eos(model, idx, eos_token_id, max_new_tokens=200, use_cache=True):
    return model.generate(idx, max_new_tokens=max_new_tokens,
                          use_cache=use_cache, eos_token_id=eos_token_id)

# 字符/BPE 示例都用换行 token；真实模型请用其配置中的 EOS id
newline_tokens = encode("\n")
if len(newline_tokens) != 1:
    raise ValueError("此示例要求换行对应单个 token；多 token 停止串需要单独检测")
eos_id = newline_tokens[0]
start = torch.zeros((1, 1), dtype=torch.long, device=device)
out = generate_until_eos(model, start, eos_id, max_new_tokens=200)
print("默认 KV cache；生成到 EOS（或长度上限）为止，长度 =", out.shape[1])
print(repr(decode(out[0].tolist())))

In [ ]:
input_text = "JACKAL:"
print("[model input]:\n\n{}\n\n=====\n\n[model output]:\n\n{}\n\n=====\n\n".format(input_text, nucleus_sampling(input_text, p = 0.3, temperature = 1, max_new_tokens = 100)[0]))

In [ ]:
input_text_list = ["JACKAL:", "Antonio:"]
for i, out in enumerate(nucleus_sampling(input_text_list, p = 0.3, temperature = 1, max_new_tokens = 100)):
    print("#{}#\n\n[model input]:\n\n{}\n\n=====\n\n[model output]:\n\n{}\n\n=====\n\n".format(i+1, input_text_list[i], out))

## 8.3. 解码之外：真实 LLM 推理还需要什么？

本节已覆盖常用的**解码策略**：greedy、temperature、top-k、top-p，并介绍 beam search 与 EOS。它们决定“下一个 token 怎么选”；要覆盖主流 **LLM 推理技术**，还需考虑执行效率、请求调度和生成控制。前文的 KV cache、GQA、MLA 正是这条线上的基础。

**先区分两个阶段**：**prefill** 一次处理 prompt 并建立 KV cache；**decode** 每步只输入新 token、读取历史 KV。长 prompt 的 prefill 通常更受算力限制，低 batch 的 decode 常受权重/KV 读取带宽限制；KV cache 消除了历史 token 的重复计算，却没有消除历史 KV 的读取。服务评估应同时看 **TTFT**（首 token 延迟，含排队与 prefill）、**TPOT**（后续每 token 的时间）、吞吐和峰值显存，优化目标取决于请求长度与并发量。

| 技术方向 | 主流做法与要点 |
| :-- | :-- |
| 请求调度 | **Continuous batching** 在每个生成步加入/移除请求；**chunked prefill** 将长 prompt 分块，减少它对 decode 的阻塞。大规模服务还可分离 prefill/decode，但需承担 KV 传输成本。 |
| KV 管理 | **Paged KV / PagedAttention** 按块分配缓存，减少预留与碎片；**prefix caching** 复用相同前缀的 KV，要求模型、token、位置与计算配置一致。GQA/MLA 则从模型结构减小缓存。 |
| 算子与运行时 | **SDPA / FlashAttention** 减少注意力中间张量和显存读写，仍是精确注意力；融合算子减少访存/启动开销。预分配 **static cache** 配合 `torch.compile` / **CUDA Graphs** 减少动态分配与调度，效果依赖形状与硬件。 |
| 低精度 | BF16/FP16 是常见执行精度；**权重量化**（INT8/INT4/FP8 等）省权重显存，**KV 量化**省长上下文缓存。加速取决于内核支持，需评估精度损失与反量化开销。 |
| 推测解码 | **Speculative decoding** 让草稿模型先提议多个 token，目标模型并行验证；正确的接受/拒绝采样可保持目标分布，但并非所有近似变体都等价。收益取决于接受率、草稿开销与负载。 |
| 多卡推理 | **TP** 切分层内矩阵，**PP** 切分模型层，MoE 的 **EP** 分布专家；通信、负载均衡与流水线空泡会影响收益，不能只看 GPU 数量。 |

**生成控制也要补齐**：repetition / presence / frequency penalty 与 no-repeat n-gram 抑制重复；**约束解码**按 grammar / JSON schema 屏蔽非法候选。对话模型要用训练时匹配的 **chat template**、BOS/EOS 与回合结束 token；多请求生成还要正确处理 padding mask、位置和每条序列的停止状态。停止字符串可能跨多个 token，不能简单等同于某个 EOS id。新一些的 min-p、typical sampling 等可按任务选用，无需为基础教程逐一实现。

这些工业级能力适合通过 [vLLM](https://docs.vllm.ai/en/latest/)、[TensorRT-LLM](https://nvidia.github.io/TensorRT-LLM/) 等框架学习；缓存类型可参考 [Transformers KV cache 文档](https://huggingface.co/docs/transformers/en/kv_cache)，推测解码原理可参考 [Fast Inference from Transformers via Speculative Decoding](https://arxiv.org/abs/2211.17192)。本教程实现常用采样与 KV cache，其余在此给出技术地图。